# Dreamer — apprendre une politique dans l'imagination latente

> **Série RL from scratch · Notebook 13 · Phase model-based : Dyna → PILCO → PETS → MBPO → _Dreamer_**

| | |
|---|---|
| **Idée** | Apprendre un **modèle de monde latent** (RSSM) puis entraîner la politique **entièrement dans l'imagination**, sans toucher l'environnement réel. |
| **Source** | [Hafner et al., *Dream to Control: Learning Behaviors by Latent Imagination* (2019, « DreamerV1 »)](https://arxiv.org/abs/1912.01603). |
| **Nouveauté vs MBPO (nb12)** | MBPO imagine de courts rollouts dans l'espace d'**observation brut**. Dreamer encode l'observation dans un **état latent compact**, y apprend la dynamique, et y rêve les trajectoires d'entraînement. |
| **Environnement** | `HalfCheetah-v5` — observation 17-dim, action 6-dim continue ∈ [−1, 1]. |

Ce notebook a une particularité : avant le RSSM, **on remonte à la source des modèles à espace d'états
(SSM)** — d'où ils viennent (théorie du contrôle), comment on les discrétise, leurs deux visages
(récursif / convolutif), et la lignée moderne (HiPPO, S4, Mamba). On construit ensuite le **RSSM de Dreamer
comme le descendant non-linéaire, stochastique, commandé et génératif** de ce SSM linéaire. L'objectif est
que cette base soit **réutilisable** : à la fin vous devez pouvoir réexpliquer chaque bloc.

## 1. Le problème que Dreamer résout

Reprenons la lignée model-based :

- **PETS** (nb[11](./11_pets_halfcheetah_walkthrough.ipynb)) apprend un *modèle de dynamique* et **planifie en ligne** par CEM à chaque pas. Excellent
  en efficacité-échantillon, mais **coûteux au moment d'agir** (replanifier des centaines de trajectoires).
- **MBPO** (nb[12](./12_mbpo_halfcheetah_walkthrough.ipynb)) remplace le planning par une **politique apprise** (SAC), entraînée sur de courts
  rollouts imaginés **dans l'espace d'observation brut**. Réponse instantanée, mais le modèle prédit
  directement $\Delta s$ et $r$ dans l'espace des 17 dimensions.

**Dreamer** pousse l'idée d'un cran : il apprend d'abord un **état latent** $s_t$ qui *résume* l'historique,
puis fait *tout* dans cet espace latent — la dynamique, la récompense, **et** l'apprentissage de la
politique. L'environnement réel ne sert plus qu'à **collecter des données** pour entraîner le modèle ;
la politique, elle, n'apprend qu'en **rêvant**.

$$
\underbrace{o_t}_{\text{observation}} \;\xrightarrow{\;\text{encodeur}\;}\; \underbrace{z_t}_{\text{état latent}}
\;\xrightarrow[\text{conditionnée par } a_t]{\;\text{dynamique latente}\;}\; z_{t+1}
\;\xrightarrow{\;\text{décodeur}\;}\; \hat o_{t+1}
$$

## 2. Pourquoi un espace latent — et l'honnêteté sur HalfCheetah

L'**encodeur** comprime l'observation en un vecteur latent compact ; le **décodeur** reconstruit
l'observation depuis ce latent. Cette compression est **décisive quand l'entrée est une image** (des
milliers de pixels → quelques dizaines de variables latentes) : c'est là que Dreamer brille (Atari, DMC
depuis pixels, Minecraft).

Sur **HalfCheetah**, l'observation est déjà un vecteur bas-dimensionnel « propre » (17 nombres physiques).
**Le gain de l'encodeur y sera donc ténu** — comprimer 17 dimensions en un latent n'apporte pas grand-chose
par rapport à des pixels. On reste néanmoins sur HalfCheetah pour deux raisons assumées :

1. **Coût de calcul** : pas de réseau convolutif lourd, un modèle léger qui tourne en quelques minutes.
2. **Continuité pédagogique** : même environnement que PETS (nb11) et MBPO (nb12), comparaisons directes.

On construit malgré tout **tout le pipeline** encodeur → latent → décodeur, car c'est la mécanique qu'on
veut comprendre en profondeur et réutiliser ailleurs (y compris sur des entrées visuelles).

## 3. Plan du notebook

- **Partie I — D'où viennent les SSM.** Théorie du contrôle, discrétisation, vues récursive/convolutive,
  lignée HiPPO/S4/Mamba. (le socle conceptuel)
- **Partie II — Le RSSM en profondeur.** L'état latent stochastique, prior vs posterior, encodeur/décodeur,
  la perte ELBO.
- **Partie III — Apprendre le comportement dans l'imagination.** Acteur/critique, λ-returns, gradient de
  valeur analytique.
- **Partie IV — Pseudocode, environnement, entraînement.**
- **Partie V — Diagnostics.**
- **Partie VI — Démo.** Reconstruction, rêve en boucle ouverte, signal de politique.
- **Parties VII-IX — Limites, ouverture DreamerV2/V3, sources.**

In [ ]:
from pathlib import Path
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.patches import Rectangle, Circle, FancyArrowPatch
from tqdm.auto import tqdm
from IPython.display import Video, display


plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
print("torch", torch.__version__)


---
# Partie I — D'où viennent les SSM

Le « RSSM » de Dreamer signifie **Recurrent State-Space Model** — *modèle récurrent à espace d'états*.
Pour le comprendre vraiment, il faut partir de l'objet dont il hérite : le **modèle à espace d'états (SSM)**
de la théorie du contrôle, vieux de plus d'un demi-siècle. C'est aussi l'objet derrière les architectures
de séquences modernes (S4, Mamba). On le construit ici de zéro, on le discrétise, on découvre ses deux
visages, puis on montre comment le RSSM le généralise.

> Référence pour approfondir la lignée SSM moderne : le blog de Loïck Bourdois,
> *Introduction aux State Space Models*, <https://lbourdois.github.io/blog/ssm/introduction_ssm/>.

## I.1 — Le modèle à espace d'états (théorie du contrôle)

Un **système à espace d'états** linéaire continu décrit comment un **état caché** $x(t)$ évolue sous une
**entrée** (commande) $u(t)$, et comment on l'**observe** via une sortie $y(t)$ :

$$
\dot x(t) = A\,x(t) + B\,u(t), \qquad y(t) = C\,x(t) + D\,u(t).
$$

- $x(t) \in \mathbb{R}^n$ — l'**état caché** : la mémoire compressée du système. Tout ce qu'il faut connaître
  du passé pour prédire le futur y est résumé (propriété de Markov *sur l'état*).
- $u(t) \in \mathbb{R}^m$ — l'**entrée / commande** : ce qu'on injecte dans le système (en RL, ce sera
  l'**action**).
- $y(t) \in \mathbb{R}^p$ — la **sortie / observation** : ce qu'on mesure (en RL, l'observation).
- $A$ (dynamique de l'état), $B$ (comment l'entrée pousse l'état), $C$ (comment l'état produit
  l'observation), $D$ (lien direct entrée→sortie, souvent nul). En deep learning on prend $D=0$.

**D'où ça vient.** Ces équations sont le langage de l'**automatique** et du **traitement du signal**
(années 1960) : asservissements, filtres, et surtout le **filtre de Kalman**, qui estime l'état caché
$x$ à partir d'observations bruitées $y$. C'est exactement le problème que le RSSM résoudra — mais en
non-linéaire et appris.

**Intuition.** L'état $x$ est une *boîte de mémoire* : un ressort-amortisseur « se souvient » de sa
position et de sa vitesse ; une fois ces deux nombres connus, son futur est déterminé par $A$ (et l'entrée
$u$). Le SSM dit : *donne-moi un état assez riche, et la dynamique du monde devient une simple récurrence
linéaire.* Le matrice $A$ encode la physique interne ; $B$ comment on agit dessus ; $C$ ce qu'on en voit.

In [ ]:
# Un SSM continu 2D : ressort-amortisseur (oscillateur amorti).
# état x = [position, vitesse].  A : dynamique interne ; B : l'entrée pousse la vitesse ; C : on observe la position.

A = np.array([[0.0, 1.0],
              [-1.0, -0.3]])   # valeurs propres complexes à partie réelle < 0  -> oscillation amortie
B = np.array([[0.0], [1.0]])
C = np.array([[1.0, 0.0]])

def simulate_continuous(A, B, C, u_fn, T=25.0, dt=1e-3, x0=None):
    n = A.shape[0]
    x = np.zeros((n, 1)) if x0 is None else x0.copy()
    ts = np.arange(0.0, T, dt)
    ys = np.empty(len(ts))
    for i, t in enumerate(ts):
        u = np.array([[u_fn(t)]])
        x = x + dt * (A @ x + B @ u)      # intégration d'Euler
        ys[i] = (C @ x)[0, 0]
    return ts, ys

u_step = lambda t: 1.0 if t >= 1.0 else 0.0     # échelon : on "pousse" à t = 1 s
ts, y = simulate_continuous(A, B, C, u_step)

def draw_ssm_block(ax):
    """Schéma-bloc du SSM : u -> B -> Σ -> ∫ -> x -> C -> y, avec rétroaction par A."""
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    YM, YF = 0.62, 0.27                                  # ligne principale / rétroaction
    def box(cx, cy, w, h, label):
        ax.add_patch(Rectangle((cx - w / 2, cy - h / 2), w, h, fc="#eef3fb", ec="k", lw=1.5, zorder=2))
        ax.text(cx, cy, label, ha="center", va="center", fontsize=15, zorder=3)
    def arrow(x0, y0, x1, y1):
        ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=14, lw=1.4, color="k", zorder=1))
    sx, sy = 0.27, YM                                    # sommateur
    ax.add_patch(Circle((sx, sy), 0.028, fc="white", ec="k", lw=1.5, zorder=2))
    ax.text(sx, sy, "+", ha="center", va="center", fontsize=14, zorder=3)
    box(0.13, YM, 0.07, 0.16, "$B$")
    box(0.45, YM, 0.09, 0.16, r"$\int$")
    box(0.71, YM, 0.07, 0.16, "$C$")
    box(0.42, YF, 0.09, 0.14, "$A$")
    ax.text(0.02, YM, "$u(t)$", ha="left", va="center", fontsize=14)
    ax.text(0.965, YM, "$y(t)$", ha="right", va="center", fontsize=14)
    arrow(0.055, YM, 0.095, YM)                          # u -> B
    arrow(0.165, YM, sx - 0.028, YM)                     # B -> Σ
    arrow(sx + 0.028, YM, 0.405, YM)                     # Σ -> ∫
    ax.text(0.35, YM + 0.07, r"$\dot{x}(t)$", ha="center", fontsize=13)
    bx = 0.585                                           # point de dérivation x(t)
    arrow(0.495, YM, 0.675, YM)                          # ∫ -> C
    ax.text(0.55, YM + 0.07, "$x(t)$", ha="center", fontsize=13)
    ax.add_patch(Circle((bx, YM), 0.007, fc="k", ec="k", zorder=4))
    arrow(0.745, YM, 0.94, YM)                           # C -> y
    ax.plot([bx, bx], [YM, YF], color="k", lw=1.4, zorder=1)   # rétroaction : x descend
    arrow(bx, YF, 0.465, YF)                             # -> A
    ax.plot([0.375, sx], [YF, YF], color="k", lw=1.4, zorder=1)
    arrow(sx, YF, sx, sy - 0.028)                        # remonte dans Σ
    ax.text(sx + 0.018, (YM + YF) / 2, "$+$", ha="left", va="center", fontsize=13)
    ax.set_title(r"Schéma-bloc du SSM : $\dot{x}=Ax+Bu,\;\; y=Cx$", fontsize=12)

fig = plt.figure(figsize=(10, 7))
gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 1.25], hspace=0.32)
ax0 = fig.add_subplot(gs[0]); draw_ssm_block(ax0)
ax1 = fig.add_subplot(gs[1])
ax1.plot(ts, y)
ax1.axvline(1.0, color="k", ls="--", lw=1, label="échelon d'entrée")
ax1.set_xlabel("temps (s)"); ax1.set_ylabel("sortie y(t)")
ax1.set_title("Réponse de l'oscillateur amorti à un échelon"); ax1.legend()
plt.show()


**Lecture.** L'entrée échelon à $t=1$ excite le système ; la sortie oscille puis s'amortit — c'est
$A$ qui dicte cette dynamique (parties réelles négatives ⇒ amortissement, parties imaginaires ⇒
oscillation). L'état $x=[\text{position}, \text{vitesse}]$ a suffi à produire tout ce comportement
temporel : c'est *ça*, la puissance d'un espace d'états bien choisi.

## I.2 — Discrétisation : du continu au discret (le point clé)

Nos données arrivent **échantillonnées** : on ne mesure pas $x(t)$ en continu, mais à des instants espacés d'un pas $\Delta$ (indice discret $k$, $t_k = k\Delta$). L'équation différentielle $\dot x = Ax + Bu$ n'est pas calculable telle quelle sur machine — il faut la transformer en une **récurrence** qui avance d'un pas à la fois :

$$
x_k = \bar A\, x_{k-1} + \bar B\, u_k, \qquad y_k = \bar C\, x_k .
$$

> *« S'il n'y a qu'une chose à retenir des SSM, c'est la discrétisation. »* (blog lbourdois)

Pourquoi est-ce *le* point clé ? Parce que c'est là qu'on choisit **comment intégrer** la dynamique — et ce choix décide de la **stabilité numérique** autant que de la fidélité au système réel.

**D'où viennent les matrices barre ?** On intègre $\dot x = Ax+Bu$ sur un pas $[t_{k-1}, t_k]$ :

$$
x_k - x_{k-1} = \int_{t_{k-1}}^{t_k} \big(Ax(t) + Bu(t)\big)\,dt .
$$

On approxime l'intégrale par la **règle des trapèzes** (moyenne des pentes aux deux bouts), en supposant $u$ ~constant sur le pas :

$$
x_k - x_{k-1} \approx \tfrac{\Delta}{2}\big(Ax_{k-1} + Ax_k\big) + \Delta\,Bu_k .
$$

On regroupe les $x_k$ à gauche, $\big(I - \tfrac{\Delta}{2}A\big)x_k = \big(I + \tfrac{\Delta}{2}A\big)x_{k-1} + \Delta\,Bu_k$, puis on inverse le premier facteur :

$$
\bar A = \Big(I - \tfrac{\Delta}{2}A\Big)^{-1}\Big(I + \tfrac{\Delta}{2}A\Big), \qquad
\bar B = \Big(I - \tfrac{\Delta}{2}A\Big)^{-1}\Delta B, \qquad \bar C = C .
$$

La barre $\bar{\;\cdot\;}$ (notation de S4) désigne le **cas discret** ; $\bar C = C$ car la sortie ne dépend pas du pas. Cette transformation a un nom : la **transformée bilinéaire** (ou de Tustin).

**Pourquoi pas la méthode d'Euler « avant » ?** Le schéma le plus naïf, $x_k = (I + \Delta A)\,x_{k-1} + \Delta B u_k$ — *celui qu'on a utilisé dans la simulation au-dessus* — est **explicite** : il extrapole avec la pente du *début* du pas. Pour un $\Delta$ trop grand il **diverge**, même quand le système continu est stable. La règle des trapèzes est **implicite** (le terme $(I-\tfrac{\Delta}{2}A)^{-1}$ fait intervenir la pente de *fin* de pas) : c'est exactement ce qui la stabilise. Géométriquement, la transformée bilinéaire envoie le **demi-plan stable** du continu ($\mathrm{Re}(\lambda)<0$) **dans le disque unité** du discret ($|z|<1$) — un système amorti le reste après discrétisation.

> Encore plus fidèle (mais plus coûteux) : le *zero-order hold* utilise l'exponentielle de matrice exacte $\bar A = e^{\Delta A}$. La transformée bilinéaire en est une très bonne approximation rationnelle, sans calcul d'`expm`.

**Le pas $\Delta$ est le curseur.** Petit $\Delta$ = fidèle au continu mais beaucoup de pas (coûteux) ; grand $\Delta$ = grossier mais rapide. C'est **le** paramètre de conception des SSM modernes : dans **S4** il est *appris* ; dans **Mamba** il devient même *dépendant de l'entrée* (le modèle décide à chaque instant « combien de temps » intégrer). On le voit à l'œuvre dès la cellule suivante.

In [ ]:
def discretize(A, B, dt):
    n = A.shape[0]; I = np.eye(n)
    inv = np.linalg.inv(I - 0.5 * dt * A)
    Ad = inv @ (I + 0.5 * dt * A)
    Bd = inv @ (dt * B)
    return Ad, Bd

def simulate_discrete(Ad, Bd, C, us):
    n = Ad.shape[0]; x = np.zeros((n, 1)); ys = np.empty(len(us))
    for k, u in enumerate(us):
        x = Ad @ x + Bd @ np.array([[u]])
        ys[k] = (C @ x)[0, 0]
    return ys

# référence "continue" très fine
ts_fine, y_fine = simulate_continuous(A, B, C, u_step, dt=1e-3)

plt.figure()
plt.plot(ts_fine, y_fine, "k", lw=2, label="continu (Δ=1e-3, référence)")
for dt in (0.5, 0.2, 0.05):
    ts_d = np.arange(0.0, 25.0, dt)
    us = np.array([u_step(t) for t in ts_d])
    Ad, Bd = discretize(A, B, dt)
    plt.plot(ts_d, simulate_discrete(Ad, Bd, C, us), "--", label=f"discret Δ={dt}")
plt.xlabel("temps (s)"); plt.ylabel("y"); plt.title("Discrétisation : la récurrence approche le continu")
plt.legend(); plt.show()

**Lecture.** Pour $\Delta$ grand (0.5) la récurrence discrète est grossière ; en raffinant ($\Delta=0.05$)
elle colle à la courbe continue. La discrétisation est le **pont** entre la physique (continue) et le calcul
(discret) — et c'est là que se joue la stabilité numérique. C'est *le* choix de conception qui distingue les
variantes de SSM modernes.

## I.3 — Les deux visages d'un SSM : récursif et convolutif

Déroulons la récurrence (avec $x_{-1}=0$, et $D=0$) en substituant chaque état dans le suivant :

$$
x_0=\bar B u_0,\quad x_1=\bar A\, x_{k_0} + \bar B\, u_1 = \bar A\bar B u_0+\bar B u_1,\quad x_2=\bar A^2\bar B u_0+\bar A\bar B u_1+\bar B u_2,\ \dots
$$

Le motif est clair : à l'instant $k$, une entrée $u_j$ (arrivée à l'instant $j\le k$) a été propagée pendant $k-j$ pas, donc multipliée par $\bar A^{\,k-j}$. En sortie $y_k=\bar C x_k$ :

$$
y_k=\sum_{j=0}^{k}\bar C\,\bar A^{\,k-j}\,\bar B\;u_j,
\qquad
\boxed{\;\bar K=\big(\bar C\bar B,\ \bar C\bar A\bar B,\ \bar C\bar A^{2}\bar B,\ \dots\big)\;}
\qquad y=\bar K * u .
$$

**Ce noyau $\bar K$ a un sens physique : c'est la réponse impulsionnelle du système.** Envoyez une impulsion unité $u=(1,0,0,\dots)$ : alors $x_0=\bar B,\ x_1=\bar A\bar B,\dots$, et la sortie vaut *exactement* $\bar K=(\bar C\bar B,\ \bar C\bar A\bar B,\dots)$. Le $i$-ème coefficient $\bar K_i=\bar C\bar A^{\,i}\bar B$ mesure « combien, $i$ pas plus tard, on voit encore l'effet d'une poussée donnée à l'instant 0 » (c'est exatement la **Réponse de l'oscillateur amorti à un échelon** vu au dessus ). Comme le système est **linéaire et invariant dans le temps (LTI)**, une entrée quelconque n'est qu'une somme d'impulsions décalées — et sa sortie, la **somme des réponses décalées** : c'est, mot pour mot, une **convolution** $y=\bar K*u$.

Donc **le même** SSM se calcule de deux façons strictement équivalentes :

- **vue récursive** : $x_k=\bar A x_{k-1}+\bar B u_k$ — séquentielle (un pas après l'autre), mémoire $O(1)$ (on ne garde que l'état courant), coût $O(L)$. Idéale en **inférence** / temps réel.
- **vue convolutive** : $y=\bar K * u$ — aucune dépendance séquentielle, donc **parallélisable** et calculable en $O(L\log L)$ par FFT. Idéale en **entraînement** sur GPU.

**Le hic — et ce que résout S4.** La vue convolutive suppose qu'on connaisse $\bar K$, qui réclame les puissances $\bar A^{0},\bar A^{1},\dots,\bar A^{L-1}$ : $L$ produits matriciels, coûteux et **numériquement instables** (cf. I.4, les valeurs propres qui explosent ou s'éteignent). Toute l'astuce de **S4** est de choisir $\bar A$ **structurée** pour calculer le noyau d'un coup, sans dérouler les puissances (via une fonction génératrice et un noyau de Cauchy).

**Pourquoi le RSSM de Dreamer, lui, n'a *qu'un* visage.** Cette double lecture n'existe **que parce que la transition est linéaire**. Dès qu'on la rend non-linéaire — le GRU du RSSM — la sortie n'est plus une convolution : plus de noyau, **seule la vue récursive subsiste**. C'est pour ça que Dreamer déroule son modèle pas à pas, tandis que les SSM linéaires (S4, Mamba) peuvent, eux, s'entraîner en parallèle.

In [ ]:
# Mini-test : la vue récursive et la vue convolutive donnent EXACTEMENT le même résultat.
Ad, Bd = discretize(A, B, dt=0.1)
rng = np.random.default_rng(0)
us = rng.standard_normal(50)

y_rec = simulate_discrete(Ad, Bd, C, us)                      # vue récursive

K = np.empty(len(us)); P = np.eye(A.shape[0])                # noyau : K[i] = C Ad^i Bd
for i in range(len(us)):
    K[i] = (C @ P @ Bd)[0, 0]
    P = Ad @ P
y_conv = np.array([np.dot(K[:k + 1][::-1], us[:k + 1]) for k in range(len(us))])  # vue convolutive

assert np.allclose(y_rec, y_conv, atol=1e-9)
print("récursif == convolutif :", np.allclose(y_rec, y_conv, atol=1e-9),
      "| écart max =", float(np.max(np.abs(y_rec - y_conv))))

**Lecture.** L'assertion passe : ce sont deux écritures du même opérateur linéaire. C'est l'astuce
centrale des **deep SSM** (S4) : on **entraîne en mode convolutif** (parallèle, rapide sur GPU) et on
**infère en mode récursif** (un pas à la fois, mémoire constante — parfait pour le temps réel).

| vue | entraînement | inférence | mémoire de contexte |
|---|---|---|---|
| continue | très lent | très lent | gère l'échantillonnage irrégulier |
| **récursive** | lent (séquentiel) | **efficace** | illimitée |
| **convolutive** | **efficace (parallèle)** | lent (autorégressif) | fixée par le noyau |

## I.4 — Pourquoi l'initialisation de $A$ est cruciale (→ [HiPPO](https://arxiv.org/pdf/2008.07669), [S4](https://arxiv.org/abs/2111.00396), [Mamba](https://arxiv.org/abs/2312.00752))

Le noyau $\bar K_i = \bar C\,\bar A^{\,i}\,\bar B$ contient les **puissances** $\bar A^{\,i}$. Or les puissances d'une matrice sont gouvernées par ses **valeurs propres** $\lambda$ : en diagonalisant $\bar A = P\Lambda P^{-1}$, on a $\bar A^{\,i} = P\Lambda^{\,i}P^{-1}$, donc chaque mode évolue comme $\lambda^{\,i}$. Tout se joue sur le **module** $|\lambda|$ :

- $|\lambda|>1 \ \Rightarrow\ \lambda^{\,i}$ **explose** (gradients qui explosent) ;
- $|\lambda|\ll 1 \ \Rightarrow\ \lambda^{\,i}$ **s'éteint** vite ⇒ le modèle **oublie** le passé (vanishing gradients).

Plus précisément, l'effet d'une entrée $u_j$ sur la sortie $i$ pas plus tard décroît comme $|\lambda|^{\,i}$ : il a chuté d'un facteur $e$ au bout d'environ $1/(1-|\lambda|)$ pas. Les valeurs propres **fixent donc l'horizon de mémoire** : juste à l'intérieur du cercle unité ($|\lambda|\lesssim 1$) = mémoire longue ; loin à l'intérieur = mémoire courte. Une init **aléatoire** de $A$ tombe presque sûrement du mauvais côté (la cellule de démo ci-dessous le visualise) — c'est exactement **la malédiction des RNN classiques** sur les dépendances longues.

**HiPPO** (*High-order Polynomial Projection Operator*) répond non par chance mais par construction. Sa matrice $A$ **structurée** est dérivée d'un objectif précis : faire en sorte que l'état $x(t)$ contienne, à chaque instant, les **coefficients de la meilleure approximation polynomiale de tout l'historique** $u(\le t)$ (projection continue sur une base de polynômes orthogonaux, p. ex. de Legendre). L'état devient une **mémoire compressée optimale du passé**, et ses valeurs propres se placent naturellement « sur le fil » — ni explosion ni oubli. En pratique, **changer seulement l'initialisation** de $A$ (aléatoire → HiPPO) fait bondir la performance (≈ 60 % → 98 % sur sequential-MNIST).

C'est le socle de la lignée moderne :
- **S4** : init HiPPO **+** un calcul *efficace* du noyau (décomposition *normal + low-rank*, noyau de Cauchy) pour éviter les $L$ puissances de I.3 ;
- **[S4D](https://arxiv.org/abs/2206.11893)** : on prend $A$ **diagonale** — bien plus simple, presque aussi performant ;
- **Mamba** : on rend $A, B, \Delta$ **dépendants de l'entrée** (*sélectifs*) — le modèle choisit, *à chaque pas*, quoi retenir ou oublier (un filtrage adaptatif au contenu), au prix de la vue convolutive, remplacée par un *scan* parallèle optimisé GPU.

**Le lien avec Dreamer.** Le RSSM n'utilise pas HiPPO : il confie ce rôle au **GRU**. Les portes du GRU (*reset* / *update*) jouent exactement le rôle des valeurs propres de $A$ — elles règlent, de façon **apprise et adaptative**, combien de passé l'état conserve. Même problème (garder une mémoire stable sur le long terme), réponses différentes : $A$ bien *initialisée* côté SSM linéaire, **gating appris** côté RNN/RSSM.

> Détails (NPLR, noyaux de Cauchy, sélectivité de Mamba) : voir le blog [lbourdois](https://lbourdois.github.io/blog/ssm/introduction_ssm/). L'essentiel : *un SSM ne mémorise correctement que si la dynamique $A$ est bien choisie — par init structurée (HiPPO) ou par gating appris (GRU).*

In [ ]:
# Démo : état d'un SSM sous dynamique pure (entrée nulle), A instable vs A stable.
def state_norms(Ad, steps=60, x0=None):
    x = np.ones((Ad.shape[0], 1)) if x0 is None else x0
    out = []
    for _ in range(steps):
        x = Ad @ x
        out.append(float(np.linalg.norm(x)))
    return out

rng = np.random.default_rng(1)
M = rng.standard_normal((2, 2))
A_unstable = M * (1.10 / max(abs(np.linalg.eigvals(M))))   # rayon spectral 1.10
A_stable   = M * (0.90 / max(abs(np.linalg.eigvals(M))))   # rayon spectral 0.90

plt.figure()
plt.semilogy(state_norms(A_unstable), label="A instable (rayon 1.10) — explose")
plt.semilogy(state_norms(A_stable),   label="A stable (rayon 0.90) — s'éteint")
plt.xlabel("pas k"); plt.ylabel("‖état‖ (échelle log)")
plt.title("L'initialisation de A décide : mémoire vs explosion / oubli")
plt.legend(); plt.show()

**Lecture.** Même tirage aléatoire, deux mises à l'échelle : au-dessus de 1 la norme **explose**, en-dessous
elle **s'éteint**. HiPPO choisit $A$ pour rester sur le fil — mémoriser sans exploser. Retenez l'idée ; les
détails analytiques sont dans le blog. On a maintenant tout le vocabulaire du SSM.

## I.5 — Du SSM linéaire au RSSM (le pont vers Dreamer)

Le SSM linéaire est élégant mais trop pauvre pour un robot qui court :

1. **dynamique linéaire** $\bar A$ → trop simple pour une physique riche (contacts, non-linéarités) ;
2. **déterministe** → ne modélise ni l'**incertitude** ni la **multimodalité** du futur ;
3. **pas de modèle génératif appris** de l'observation (un $C$ linéaire ne sait pas reconstruire une image).

Dreamer — plus précisément le **RSSM**, introduit par [PlaNet (Hafner et al., 2019)](https://arxiv.org/abs/1811.04551) et réutilisé par [Dreamer (Hafner et al., 2019)](https://arxiv.org/abs/1912.01603) — garde l'idée maîtresse (*un état latent qui résume le passé, évolue sous une commande, émet une observation*) mais remplace chaque brique linéaire par une version **apprise**. Comme cela introduit plusieurs objets nouveaux, voici de quoi il s'agit avant le détail de la Partie II.

**La transition $\bar A$ → un GRU (récurrence non-linéaire).** Un **GRU** (*Gated Recurrent Unit*, [Cho et al., 2014](https://arxiv.org/abs/1406.1078)) est un RNN muni de **portes apprises** : une porte de *mise à jour* décide quelle fraction de l'ancien état conserver, une porte de *reset* quelle fraction oublier avant d'incorporer la nouvelle entrée. Ces portes jouent le rôle que les valeurs propres de $\bar A$ jouaient en I.4 — régler l'horizon de mémoire — mais de façon **non-linéaire et adaptative** :
$$h_t=\mathrm{GRU}\big(h_{t-1},\,[z_{t-1},a_{t-1}]\big). \qquad \text{(détaillé en II.4)}$$

**L'entrée $u$ → l'action $a_t$.** La « commande » du système est désormais l'**action** choisie par la politique : c'est par là que le RL pilote la dynamique latente.

**L'émission $\bar C x$ → un décodeur + une tête récompense.** L'émission linéaire devient un **décodeur** (MLP ici, CNN sur pixels) — un vrai **modèle génératif** $p(o_t\mid s_t)$ qui *reconstruit* l'observation depuis le latent, l'analogue appris et non-linéaire de $C$. On y ajoute une **tête récompense** $\hat r_t=\mathrm{rew}(s_t)$, indispensable pour **évaluer des trajectoires imaginées**, où la vraie récompense de l'environnement n'est pas disponible. *(détaillés en II.5 et II.6)*

**Nouveau : un état stochastique $z_t$ + filtrage variationnel.** En plus de la mémoire déterministe $h_t$, le RSSM tire une variable **gaussienne** $z_t$ qui capture l'**incertitude** (plusieurs futurs plausibles). Elle vient en deux versions, exactement comme un **filtre de Kalman** ([Kalman, 1960](https://www.cs.unc.edu/~welch/kalman/media/pdf/Kalman1960.pdf)) :
- le **prior** $p(z_t\mid h_t)$ — étape *predict* : ce que le modèle prédit *sans* observation (servira à imaginer) ;
- le **posterior** $q(z_t\mid h_t, e_t)$ — étape *update* : la correction *après* avoir vu l'observation encodée $e_t$.

Le filtre de Kalman classique fait ce predict/update **analytiquement**, pour un système **linéaire-gaussien** ; le RSSM le fait avec des **réseaux**, sur une dynamique **non-linéaire** — d'où « filtre de Kalman appris ». On parle de filtrage **variationnel** parce que le posterior $q$ est une *approximation* optimisée via une borne (ELBO, détaillée en II.7), avec l'astuce de **reparamétrisation** des [VAE (Kingma & Welling, 2013)](https://arxiv.org/abs/1312.6114) pour rester différentiable.

**Table de correspondance — à mémoriser.**

| SSM linéaire (contrôle) | RSSM (Dreamer) | rôle |
|---|---|---|
| état $x_t$ | $s_t=[h_t,z_t]$ (déterministe + stochastique) | mémoire latente du passé |
| transition $x_t=\bar A x_{t-1}+\bar B u_t$ | $h_t=\mathrm{GRU}(h_{t-1},[z_{t-1},a_{t-1}])$ | dynamique (linéaire → non-linéaire) |
| entrée / commande $u_t$ | **action** $a_t$ | ce qu'on injecte |
| émission $y_t=\bar C x_t$ | **décodeur** $\hat o_t=\mathrm{dec}(s_t)$ + tête $\hat r_t$ | observation reconstruite / récompense |
| (rien) | $z_t\sim q(z_t\mid h_t,e_t)$ vs $p(z_t\mid h_t)$ | incertitude + filtrage (≈ Kalman) |
| $A$ bien initialisée ([HiPPO](https://arxiv.org/abs/2008.07669)) | GRU + KL prior↔posterior | stabilité de la mémoire |

C'est **le même squelette** qu'un SSM, rendu non-linéaire, stochastique, commandé et génératif.

**Récapitulatif Partie I.** Un SSM, c'est un **état latent** qui résume le passé, **évolue sous une entrée**, et **émet une observation** ; on le **discrétise** en récurrence ; il a deux visages **récursif/convolutif** ; et tout dépend de l'**initialisation de la dynamique** ([HiPPO](https://arxiv.org/abs/2008.07669) → [S4](https://arxiv.org/abs/2111.00396) → [Mamba](https://arxiv.org/abs/2312.00752)). Le **RSSM de Dreamer** en est la version apprise et non-linéaire. Construisons-le.

---
# Partie II — Le RSSM, le modèle de monde de Dreamer

## II.1 — L'état $s_t=[h_t,z_t]$ : mémoire déterministe et variable stochastique

Le **Recurrent State-Space Model** ne représente pas le monde avec un unique vecteur latent. Son état
interne contient deux parties complémentaires :

$$
s_t = [h_t,z_t].
$$

- $h_t$ est l'état **déterministe** porté par un GRU. À actions et états précédents identiques, il produit
  toujours la même valeur. Il sert de **mémoire temporelle** : il résume ce que le modèle a retenu de
  l'histoire passée.
- $z_t$ est un état **stochastique**, ici échantillonné depuis une gaussienne. Il représente l'information
  nouvelle ou incertaine qui ne peut pas être décrite par la seule mémoire déterministe.

Leur évolution n'a donc pas exactement le même rôle :

$$
h_t=\operatorname{GRU}\!\left(h_{t-1},[z_{t-1},a_{t-1}]\right),
\qquad
z_t\sim p(z_t\mid h_t)
\quad\text{ou}\quad
z_t\sim q(z_t\mid h_t,e_t).
$$

Le GRU met d'abord à jour $h_t$ à partir de la mémoire précédente, du latent précédent et de l'action
exécutée. Une distribution sur $z_t$ est ensuite construite à partir de cette nouvelle mémoire. Nous
verrons dans la section suivante pourquoi il en existe deux : le **prior**, qui prédit sans voir
l'observation, et le **posterior**, qui peut corriger cette prédiction après l'avoir vue.

### Pourquoi ne pas utiliser seulement $h_t$ ?

Un état entièrement déterministe devrait produire un futur unique. Or, avec une observation partielle ou
ambiguë, plusieurs états réels peuvent être compatibles avec le même historique observable. Forcer le
réseau à choisir une seule possibilité conduit souvent à une moyenne peu réaliste ou à un modèle
excessivement certain.

À l'inverse, un état uniquement stochastique devrait réencoder toute l'histoire à chaque pas et aurait
davantage de difficulté à conserver une mémoire stable sur de longues séquences.

Une analogie utile est celle d'un **carnet de bord** :

- $h_t$ contient le résumé déjà établi de tout ce qui s'est passé ;
- $z_t$ représente l'hypothèse courante sur les détails du présent que ce résumé ne suffit pas à fixer.

Le déterministe apporte donc la **continuité temporelle** ; le stochastique apporte de la **souplesse
représentationnelle** et permet d'échantillonner plusieurs trajectoires latentes plausibles. Une gaussienne
diagonale reste néanmoins une approximation simple : elle ne représente pas, à elle seule, toutes les
formes possibles d'incertitude ou de multimodalité.

Les autres composants de Dreamer ne travaillent généralement pas directement avec $h_t$ ou $z_t$ pris
isolément. Ils utilisent leur concaténation :

$$
\operatorname{feat}(s_t)=[h_t,z_t].
$$

Cette feature alimente le décodeur, la tête de récompense, l'acteur et le critique. Elle rassemble ainsi
ce que le modèle **mémorise** et ce qu'il **suppose actuellement** sur l'état du monde.

## II.2 — Prior et posterior : prédire, puis corriger

À chaque pas, le RSSM construit deux distributions gaussiennes sur le même latent $z_t$ :

$$
\underbrace{p(z_t\mid h_t)}_{\text{prior : sans observation}}
\qquad\text{et}\qquad
\underbrace{q(z_t\mid h_t,e_t)}_{\text{posterior : avec }e_t=\operatorname{enc}(o_t)}.
$$

Ces distributions n'ont pas deux rôles concurrents. Elles correspondent à deux niveaux d'information.

### Le prior : ce que le modèle prévoit

Le **prior** ne reçoit que la mémoire déterministe $h_t$. Il répond à la question :

> « D'après le passé et l'action précédente, dans quel état latent devrais-je maintenant me trouver ? »

Il produit une moyenne et un écart-type :

$$
p(z_t\mid h_t)=\mathcal N\!\left(\mu_t^p,(\sigma_t^p)^2I\right).
$$

C'est la seule distribution disponible lorsque Dreamer déroule une trajectoire dans son imagination :
aucune nouvelle observation réelle n'existe alors pour corriger la prédiction.

### Le posterior : ce que le modèle estime après observation

Pendant la collecte et l'apprentissage, l'observation réelle $o_t$ est disponible. L'encodeur la transforme
en embedding $e_t$, puis le **posterior** combine cette nouvelle information avec $h_t$ :

$$
q(z_t\mid h_t,e_t)=\mathcal N\!\left(\mu_t^q,(\sigma_t^q)^2I\right).
$$

Il répond à une question légèrement différente :

> « Maintenant que j'ai réellement observé le monde, quelle est ma meilleure estimation de son état latent ? »

Le posterior joue donc le rôle d'une **correction informée**. Il peut déplacer la moyenne prévue par le
prior ou modifier son incertitude lorsque l'observation révèle quelque chose d'inattendu.

### L'analogie avec le filtre de Kalman

Le mécanisme suit le même rythme que le filtre de Kalman :

1. **predict** : faire avancer la croyance à l'aide de la dynamique ;
2. **update** : corriger cette croyance avec une nouvelle mesure.

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    prev["état précédent<br/>$$~h_{t-1},z_{t-1}$$"]
    action["action précédente<br/>$$~a_{t-1}$$"]
    memory["mémoire mise à jour<br/>$$~h_t$$"]
    prior["PREDICT<br/>$$~p(z_t\mid h_t)$$"]
    obs["observation réelle<br/>$$~o_t$$"]
    encoder["encodeur<br/>$$~e_t=\mathrm{enc}(o_t)$$"]
    posterior["UPDATE<br/>$$~q(z_t\mid h_t,e_t)$$"]
    kldiv["la KL divergence rapproche prior et posterior <br/>$$~\operatorname{KL}(q\Vert p)$$"]

    prev --> memory
    action --> memory
    memory --> prior
    memory --> posterior
    obs --> encoder --> posterior
    prior --> kldiv
    posterior --> kldiv
```

La différence est que le filtre de Kalman effectue cette correction analytiquement dans un système
linéaire-gaussien connu. Le RSSM **apprend** les étapes de prédiction et de correction avec des réseaux
de neurones dans un système non linéaire.

Une autre analogie est celle d'un navigateur :

- le prior est sa position estimée à partir de la vitesse et du cap ;
- le posterior est sa position corrigée lorsqu'il reçoit enfin une mesure GPS.

Pendant un rêve, le GPS est coupé : Dreamer doit naviguer avec le prior seul.

### Apprendre au prior à prédire sans regarder

Au début de l'entraînement, le posterior est généralement meilleur : il a accès à l'observation réelle,
contrairement au prior. On utilise donc le posterior comme estimation de référence et on rapproche le
prior de celui-ci avec :

$$
\operatorname{KL}\!\left(
q(z_t\mid h_t,e_t)\,\Vert\,p(z_t\mid h_t)
\right).
$$

Cette divergence pénalise un prior qui attribue peu de probabilité aux latents jugés plausibles par le
posterior. Progressivement, le prior apprend à reproduire, depuis la seule mémoire $h_t$, l'information que
le posterior obtenait grâce à l'observation.

L'objectif n'est toutefois pas de rendre les deux distributions identiques à tout prix. Si la pression KL
est trop forte, le modèle peut choisir une solution triviale : rendre $z_t$ presque indépendant de
l'observation. Le décodeur s'appuie alors principalement sur $h_t$ et le latent stochastique devient inutile.
C'est le **posterior collapse**.

Dreamer introduit pour cela les **free nats** :

$$
\mathcal L_{\mathrm{KL}}
=
\max\!\left(\text{free\_nats},
\operatorname{KL}(q\Vert p)\right).
$$

En dessous du seuil, une petite quantité d'information latente est considérée comme « gratuite » et ne
subit plus de pression supplémentaire. Le modèle peut ainsi utiliser $z_t$ pour encoder des informations
utiles avant que la régularisation ne cherche à rapprocher davantage prior et posterior.

Le compromis est essentiel :

- le posterior doit rester assez informatif pour représenter correctement les observations ;
- le prior doit devenir assez proche du posterior pour produire des rêves crédibles sans observation.

On entraînera le prior à **se rapprocher** du posterior (terme KL). À terme, le modèle sait prédire le futur
*sans* regarder — condition nécessaire pour apprendre une politique purement dans l'imagination.

## II.3 — L'encodeur : transformer l'observation en preuve latente

Le posterior ne reçoit pas directement l'observation brute $o_t$. Celle-ci passe d'abord par un encodeur :

$$
e_t=\operatorname{enc}_\theta(o_t).
$$

L'embedding $e_t$ n'est pas encore l'état latent complet de Dreamer. Il constitue plutôt une **preuve
compacte extraite de l'observation**, que le posterior combine avec la mémoire $h_t$ pour estimer $z_t$.

Cette distinction est importante :

- l'encodeur décrit principalement **ce que l'on voit maintenant** ;
- le GRU conserve **ce que l'on sait du passé** ;
- le posterior fusionne les deux pour former l'état latent courant.

Sur des observations visuelles, l'encodeur est généralement un CNN : il transforme des milliers de pixels
en caractéristiques compactes comme des formes, des positions ou des mouvements. Cette compression évite
au RSSM de prédire directement dans l'immense espace des pixels.

Ici, HalfCheetah fournit déjà un vecteur de seulement 17 variables physiques. Un MLP suffit donc :

$$
o_t\in\mathbb R^{17}
\;\longrightarrow\;
e_t\in\mathbb R^{d_e}.
$$

Le bénéfice de compression est moins spectaculaire que sur des images. L'encodeur conserve néanmoins un
rôle utile : il apprend une représentation adaptée au RSSM et permet de garder exactement la même
architecture conceptuelle que Dreamer sur pixels.

On peut voir l'encodeur comme un **traducteur** : l'environnement parle en positions, angles et vitesses,
tandis que le RSSM travaille dans son propre langage latent. L'encodeur traduit l'observation dans ce
langage ; le décodeur, construit plus loin, effectuera le trajet inverse.

In [ ]:
class Encoder(nn.Module):
    """Comprime l'observation (17-dim) en un embedding compact."""
    def __init__(self, obs_dim, hidden_dim, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, embed_dim), nn.SiLU(),
        )
    def forward(self, obs):
        return self.net(obs)

# Mini-test : shapes
_enc = Encoder(obs_dim=17, hidden_dim=32, embed_dim=16)
_o = torch.randn(4, 17)
assert _enc(_o).shape == (4, 16)
print("Encoder OK :", tuple(_enc(_o).shape))

## II.4 — Le RSSM : GRU + prior + posterior

Au cœur, une cellule récurrente et deux têtes gaussiennes. Pour chaque pas :

1. $x = \mathrm{MLP}([z_{t-1}, a_{t-1}])$ puis $h_t = \mathrm{GRUCell}(x, h_{t-1})$ — la **dynamique** ;
2. **prior** : $(\mu^p,\sigma^p)=\mathrm{MLP}(h_t)$ ;
3. **posterior** : $(\mu^q,\sigma^q)=\mathrm{MLP}([h_t, e_t])$ ;
4. échantillonnage **reparamétré** $z = \mu+\sigma\,\varepsilon$ (gradient préservé).

`obs_step` renvoie (posterior, prior) ; `img_step` ne renvoie que le prior (imagination, sans obs).

**Le RSSM en un schéma** (un pas) — `obs_step` utilise l'observation (chemin plein), `img_step` n'utilise que le prior (chemin pointillé, pour l'imagination) :

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    obs["observation $$~o_t$$"]
    act["action $$~a_{t-1}$$"]
    prev["état précédent $$~h_{t-1}, z_{t-1}$$"]
    enc["encodeur $$~e_t$$"]
    gru["GRUCell<br/>$$~h_t$$"]
    prior["prior (predict)<br/>$$~p(z_t \mid h_t)$$"]
    post["posterior (update)<br/>$$~q(z_t \mid h_t, e_t)$$"]
    feat["feature $$~s_t = [h_t, z_t]$$"]
    dec["décodeur $$~\hat o_t$$"]
    rew["récompense $$~\hat r_t$$"]
    prev --> gru
    act --> gru
    gru --> prior
    gru --> post
    obs --> enc
    enc --> post
    post --> feat
    prior -. "img_step (sans obs)" .-> feat
    feat --> dec
    feat --> rew
```

In [ ]:
def _gauss(net, head, x, min_std=0.1):
    h = net(x)
    mean, raw = head(h).chunk(2, dim=-1)
    std = F.softplus(raw) + min_std
    return mean, std

class RSSM(nn.Module):
    """Recurrent State-Space Model : h déterministe (GRU) + z stochastique (gaussien)."""
    def __init__(self, action_dim, embed_dim, deter_dim, stoch_dim, hidden_dim, min_std=0.1):
        super().__init__()
        self.deter_dim, self.stoch_dim, self.min_std = deter_dim, stoch_dim, min_std
        self.in_proj   = nn.Sequential(nn.Linear(stoch_dim + action_dim, hidden_dim), nn.SiLU())
        self.cell      = nn.GRUCell(hidden_dim, deter_dim)
        self.prior_net = nn.Sequential(nn.Linear(deter_dim, hidden_dim), nn.SiLU())
        self.prior_h   = nn.Linear(hidden_dim, 2 * stoch_dim)
        self.post_net  = nn.Sequential(nn.Linear(deter_dim + embed_dim, hidden_dim), nn.SiLU())
        self.post_h    = nn.Linear(hidden_dim, 2 * stoch_dim)

    def initial_state(self, batch):
        z = torch.zeros(batch, self.stoch_dim); h = torch.zeros(batch, self.deter_dim)
        return {"deter": h, "stoch": z, "mean": z.clone(), "std": torch.ones_like(z)}

    def obs_step(self, prev, prev_action, embed):
        x = self.in_proj(torch.cat([prev["stoch"], prev_action], -1))
        h = self.cell(x, prev["deter"])
        pm, ps = _gauss(self.prior_net, self.prior_h, h, self.min_std)
        qm, qs = _gauss(self.post_net,  self.post_h,  torch.cat([h, embed], -1), self.min_std)
        prior = {"deter": h, "stoch": pm + ps * torch.randn_like(ps), "mean": pm, "std": ps}
        post  = {"deter": h, "stoch": qm + qs * torch.randn_like(qs), "mean": qm, "std": qs}
        return post, prior

    def img_step(self, prev, prev_action):
        x = self.in_proj(torch.cat([prev["stoch"], prev_action], -1))
        h = self.cell(x, prev["deter"])
        pm, ps = _gauss(self.prior_net, self.prior_h, h, self.min_std)
        return {"deter": h, "stoch": pm + ps * torch.randn_like(ps), "mean": pm, "std": ps}

    def get_feat(self, s):
        return torch.cat([s["deter"], s["stoch"]], -1)

    def kl(self, post, prior, free_nats):
        q = torch.distributions.Normal(post["mean"], post["std"])
        p = torch.distributions.Normal(prior["mean"], prior["std"])
        kl = torch.distributions.kl_divergence(q, p).sum(-1)        # KL(post || prior)
        return torch.clamp(kl, min=free_nats).mean()

# Mini-test : shapes de obs_step / img_step et KL >= free_nats
_rssm = RSSM(action_dim=6, embed_dim=16, deter_dim=32, stoch_dim=8, hidden_dim=32)
_s = _rssm.initial_state(4)
_post, _prior = _rssm.obs_step(_s, torch.zeros(4, 6), torch.randn(4, 16))
assert _rssm.get_feat(_post).shape == (4, 32 + 8)
assert _rssm.img_step(_s, torch.zeros(4, 6))["stoch"].shape == (4, 8)
assert float(_rssm.kl(_post, _prior, free_nats=3.0)) >= 3.0 - 1e-5
print("RSSM OK | feat", tuple(_rssm.get_feat(_post).shape), "| KL>=free_nats ✓")

## II.5 — Le décodeur : reconstruire les 17 dimensions

Le décodeur prend la feature latente $s_t$ et **reconstruit l'observation** $\hat o_t$. C'est l'équivalent
non linéaire de la matrice $\bar C$ du SSM : il traduit la représentation interne de Dreamer vers les
positions, angles et vitesses observables de HalfCheetah. On modélise $p(o_t\mid s_t)$ par une gaussienne
de variance unité, ce qui revient à minimiser une simple **MSE**. Cette reconstruction force ainsi l’état
latent à conserver suffisamment d’information sur le monde réel, plutôt qu’une représentation abstraite
mais inutilisable.

In [ ]:
class Decoder(nn.Module):
    """Reconstruit l'observation 17-dim depuis la feature latente (rôle de C dans un SSM)."""
    def __init__(self, feat_dim, hidden_dim, obs_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, obs_dim),
        )
    def forward(self, feat):
        return self.net(feat)

# Mini-test
_dec = Decoder(feat_dim=40, hidden_dim=32, obs_dim=17)
assert _dec(torch.randn(4, 40)).shape == (4, 17)
print("Decoder OK")

## II.6 — La tête récompense

Comme MBPO, on apprend à **prédire la récompense** directement depuis l’état latent. Cette tête apprend
quelles configurations latentes correspondent à des situations favorables ou défavorables. Elle est
indispensable pour évaluer les trajectoires *imaginées*, où l’environnement réel n’est pas présent pour
fournir la récompense : Dreamer doit pouvoir estimer lui-même la qualité de ses rêves.

In [ ]:
class RewardModel(nn.Module):
    """Prédit la récompense scalaire depuis la feature latente."""
    def __init__(self, feat_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )
    def forward(self, feat):
        return self.net(feat).squeeze(-1)

# Mini-test
_rm = RewardModel(feat_dim=40, hidden_dim=32)
assert _rm(torch.randn(4, 40)).shape == (4,)
print("RewardModel OK")

## II.7 — La perte du modèle de monde : l’ELBO

On entraîne conjointement l’encodeur, le RSSM, le décodeur et la tête de récompense sur des séquences
$(o_{1:L},a_{1:L},r_{1:L})$. L’objectif est que l’état latent explique les observations et les
récompenses, tout en restant prévisible par la dynamique du prior.

### Étape 1 — Le problème de vraisemblance

On voudrait maximiser la probabilité des données observées,
$\log p(o_{1:L},r_{1:L}\mid a_{1:L})$. Cependant, les états $z_{1:L}$ sont cachés : calculer cette
probabilité exige de sommer ou d’intégrer sur **toutes les trajectoires latentes possibles**, ce qui est
intractable avec des réseaux non linéaires.

On introduit donc le posterior approché $q(z_t\mid h_t,e_t)$, qui propose des états latents plausibles
après avoir observé $o_t$. En multipliant et divisant l’intégrande par $q$, puis en appliquant l’inégalité
de Jensen au logarithme d’une espérance, on obtient une borne calculable :

$$
\begin{aligned}
\log p(o_{1:L},r_{1:L}\mid a_{1:L})
&=
\log\mathbb E_{q(z_{1:L})}
\left[
\frac{p(o_{1:L},r_{1:L},z_{1:L}\mid a_{1:L})}
     {q(z_{1:L}\mid o_{1:L},a_{1:L})}
\right] \\[2mm]
&\ge
\sum_{t=1}^{L}
\Big[
\underbrace{\mathbb E_q[\log p(o_t\mid s_t)]}_{\text{expliquer l’observation}}
+
\underbrace{\mathbb E_q[\log p(r_t\mid s_t)]}_{\text{expliquer la récompense}}
-
\underbrace{\operatorname{KL}\!\left(q(z_t\mid h_t,e_t)\,\Vert\,p(z_t\mid h_t)\right)}
_{\text{rendre le latent prévisible}}
\Big].
\end{aligned}
$$

Cette quantité est l’**Evidence Lower Bound**, ou **ELBO** : une borne inférieure de la
log-vraisemblance. On ne maximise donc pas directement la vraie vraisemblance, mais une quantité
calculable qui lui est inférieure. Améliorer cette borne pousse néanmoins le modèle à mieux expliquer
les séquences observées.

### Étape 2 — Passer de l’ELBO à une perte

Nous modélisons l’observation et la récompense par des gaussiennes de variance unité. Le négatif de leur
log-vraisemblance devient alors, à une constante près, une erreur quadratique. Comme les optimiseurs
PyTorch **minimisent** une perte, on change également le signe de l’ELBO :

$$
\boxed{
\mathcal L_{\mathrm{world}}
=
\sum_{t=1}^{L}
\left[
\underbrace{\frac12\lVert\hat o_t-o_t\rVert^2}_{\mathcal L_{\mathrm{obs}}}
+
\underbrace{\frac12(\hat r_t-r_t)^2}_{\mathcal L_{\mathrm{reward}}}
+
\beta\,
\underbrace{
\max\!\left(
\mathrm{free\_nats},
\operatorname{KL}\!\left(q_t\Vert p_t\right)
\right)
}_{\mathcal L_{\mathrm{KL}}}
\right].
}
$$

Les trois termes ont des responsabilités différentes :

- **Reconstruction $\mathcal L_{\mathrm{obs}}$.** Elle force $s_t=[h_t,z_t]$ à conserver assez
  d’information pour reconstruire les 17 dimensions de HalfCheetah. Sans elle, le latent pourrait
  oublier des éléments importants de la dynamique physique.
- **Récompense $\mathcal L_{\mathrm{reward}}$.** Elle organise le latent autour de ce qui compte pour le
  contrôle. Deux états visuellement proches mais conduisant à des récompenses différentes doivent pouvoir
  être distingués.
- **KL $\mathcal L_{\mathrm{KL}}$.** Le posterior a vu l’observation réelle ; le prior ne dispose que de
  la mémoire. La KL apprend au prior à reproduire la croyance du posterior afin que les trajectoires
  générées sans observation restent plausibles.
- **Coefficient $\beta$.** Dans le code, `kl_scale` règle le compromis entre fidélité aux données et
  prévisibilité du latent. Trop faible, le prior ne suit pas le posterior ; trop élevé, le modèle peut
  sacrifier l’information portée par $z_t$.
- **Free nats.** Tant que la KL reste sous ce budget, sa réduction n’est pas récompensée. Le posterior
  peut donc utiliser une petite quantité d’information sans être immédiatement forcé de ressembler au
  prior. Cela limite le risque de **posterior collapse**, où $z_t$ devient inutile et toute la
  représentation repose sur $h_t$.

On peut résumer l’équilibre ainsi : la reconstruction et la récompense demandent au latent d’être
**informatif**, tandis que la KL lui demande d’être **prévisible**. Un bon modèle de monde doit satisfaire
les deux : représenter fidèlement ce qui s’est réellement passé, puis être capable de le prolonger sans
nouvelle observation.

In [ ]:
def observe_sequence(encoder, rssm, decoder, reward_model, obs, action, free_nats=1.0, kl_scale=1.0):
    """Déroule le posterior sur une séquence et calcule la perte du modèle de monde.

    obs    : [B, L, obs_dim]
    action : [B, L, action_dim]   (action[t] = action prise APRÈS obs[t])
    """
    B, L, _ = obs.shape
    embed = encoder(obs)                                            # [B, L, E]
    prev_actions = torch.cat([torch.zeros_like(action[:, :1]), action[:, :-1]], dim=1)  # a_{t-1}
    state = rssm.initial_state(B)
    feats, kls = [], []
    for t in range(L):
        post, prior = rssm.obs_step(state, prev_actions[:, t], embed[:, t])
        feats.append(rssm.get_feat(post))
        kls.append(rssm.kl(post, prior, free_nats))
        state = post
    feat = torch.stack(feats, dim=1)                               # [B, L, F]
    recon_loss = 0.5 * ((decoder(feat) - obs) ** 2).sum(-1).mean()
    return feat, recon_loss, torch.stack(kls).mean()

# Mini-test + visualisation : sur une séquence-jouet, le modèle apprend à comprimer puis reconstruire.
torch.manual_seed(0)
enc = Encoder(5, 32, 16); rssm = RSSM(2, 16, 32, 8, 32); dec = Decoder(40, 32, 5); rm = RewardModel(40, 32)
opt = torch.optim.Adam(list(enc.parameters()) + list(rssm.parameters()) + list(dec.parameters()), lr=3e-3)

# séquences : sinusoïdes lentes bruitées (5 capteurs déphasés), 6 séquences de longueur 20
t = torch.linspace(0, 6.28, 20)
toy_obs = torch.stack([torch.sin(t[None, :] + p) for p in torch.linspace(0, 3.0, 5)], dim=-1)
toy_obs = toy_obs.repeat(6, 1, 1) + 0.02 * torch.randn(6, 20, 5)
toy_act = torch.zeros(6, 20, 2)

recon_hist = []
for it in range(80):
    _, recon, kl = observe_sequence(enc, rssm, dec, rm, toy_obs, toy_act, free_nats=0.5)
    loss = recon + kl
    opt.zero_grad(); loss.backward(); opt.step()
    recon_hist.append(recon.item())
assert recon_hist[-1] < recon_hist[0] * 0.6
print(f"recon: {recon_hist[0]:.3f} -> {recon_hist[-1]:.3f}")

with torch.no_grad():
    feat0, _, _ = observe_sequence(enc, rssm, dec, rm, toy_obs[:1], toy_act[:1], free_nats=0.5)
    rec0 = dec(feat0)[0].numpy()
real0 = toy_obs[0].numpy()
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].plot(recon_hist); ax[0].set_title("perte de reconstruction"); ax[0].set_xlabel("pas de gradient")
for d, a in zip([0, 2], ax[1:]):
    a.plot(real0[:, d], "k", label="réel"); a.plot(rec0[:, d], "--", label="reconstruit")
    a.set_title(f"capteur {d}"); a.set_xlabel("pas")
ax[1].legend(fontsize=8); fig.suptitle("Le modèle de monde apprend à comprimer puis reconstruire")
fig.tight_layout(); plt.show()

**Lecture.** La perte de reconstruction chute nettement : l'encodeur, le RSSM et le décodeur ont appris à
**comprimer puis reconstruire** la séquence, et le KL a aligné prédiction et filtrage. Le modèle de monde
fonctionne — on peut maintenant *l'habiter*.

## II.7bis — Prior vs posterior, rendu visible

Sur le même modèle-jouet, comparons les deux distributions sur $z_t$ : le **prior** (prédiction depuis la seule mémoire $h_t$, *avant* l'observation) et le **posterior** (correction *après* avoir vu $o_t$). On décode chacun et on suit le KL par pas — c'est le cycle *predict / update* d'un filtre de Kalman.

In [ ]:
# Prior (predict, sans obs) vs posterior (update, avec obs) sur la séquence-jouet
with torch.no_grad():
    obs1, act1 = toy_obs[:1], toy_act[:1]
    embed1 = enc(obs1)
    prev_a1 = torch.cat([torch.zeros_like(act1[:, :1]), act1[:, :-1]], 1)
    state = rssm.initial_state(1)
    dec_post, dec_prior, kl_step = [], [], []
    for tt in range(obs1.shape[1]):
        post, prior = rssm.obs_step(state, prev_a1[:, tt], embed1[:, tt])
        dec_post.append(dec(rssm.get_feat(post))[0].numpy())
        dec_prior.append(dec(rssm.get_feat(prior))[0].numpy())
        kl_step.append(float(torch.distributions.kl_divergence(
            torch.distributions.Normal(post["mean"], post["std"]),
            torch.distributions.Normal(prior["mean"], prior["std"])).sum().item()))
        state = post
    dec_post, dec_prior = np.array(dec_post), np.array(dec_prior)
real0 = toy_obs[0].numpy()

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(real0[:, 0], "k", label="réel")
ax[0].plot(dec_post[:, 0], "--", label="posterior (voit l'obs)")
ax[0].plot(dec_prior[:, 0], ":", label="prior (prédit avant l'obs)")
ax[0].set_title("Décodage capteur 0"); ax[0].set_xlabel("pas"); ax[0].legend(fontsize=8)
ax[1].plot(kl_step, marker="."); ax[1].set_title("KL(posterior ‖ prior) par pas"); ax[1].set_xlabel("pas")
fig.suptitle("Prior (predict) vs Posterior (update) — filtrage à la Kalman")
fig.tight_layout(); plt.show()

**Lecture.** Le posterior voit l'observation courante et doit donc suivre le signal réel plus fidèlement ; le prior ne dispose que de la mémoire et représente la prédiction utilisée pour rêver. La KL mesure précisément la correction apportée par l'observation à chaque pas. Si elle reste durablement grande, le modèle reconstruit peut-être correctement en filtrage, mais son prior n'est pas encore assez fiable pour des trajectoires imaginées en boucle ouverte.

---
# Partie III — Apprendre une politique en rêvant

## III.1 — L'idée : rêver des trajectoires, y entraîner l'acteur

Une fois le modèle de monde appris, on n'a **plus besoin de l'environnement réel** pour améliorer la
politique. On part d'états réels (les posteriors), puis on **déroule la dynamique latente** (`img_step`,
prior seul) avec l'acteur sur $H$ pas, en prédisant la récompense à chaque pas. Ces trajectoires
**imaginées** sont gratuites et illimitées — on y entraîne acteur et critique.

C'est la version *latente* et *profonde* de Dyna ([nb09](./09_dyna_q_dyna_q_plus_deep_dyna_walkthrough.ipynb)) : générer des données dans la tête du modèle.

## III.2 — Acteur et critique (rappel → [nb08](./08_sac_halfcheetah_walkthrough.ipynb))

Dreamer apprend son comportement avec un couple **acteur–critique**, comme les méthodes vues précédemment.
La différence essentielle est que leurs données ne viennent pas directement de l’environnement : elles
proviennent des trajectoires **imaginées dans le RSSM**. Tous deux reçoivent la feature latente
$\operatorname{feat}(s_t)=[h_t,z_t]$, et non l’observation brute.

### L’acteur : choisir une action continue

L’acteur $\pi_\phi(a_t\mid s_t)$ produit les paramètres d’une gaussienne, puis échantillonne une action
non bornée $u_t$. Un `tanh` la ramène dans l’intervalle autorisé $[-1,1]$ :

$$
\mu_\phi(s_t),\sigma_\phi(s_t)=\operatorname{Actor}_\phi(s_t),
\qquad
u_t=\mu_\phi(s_t)+\sigma_\phi(s_t)\odot\varepsilon,
\qquad
a_t=\tanh(u_t),
\quad \varepsilon\sim\mathcal N(0,I).
$$

L’écriture $u_t=\mu+\sigma\varepsilon$ est l’astuce de **reparamétrisation**. L’aléatoire est isolé dans
$\varepsilon$, tandis que l’action reste une fonction différentiable des paramètres $\phi$. Le gradient
peut ainsi suivre toute la chaîne :

```text
paramètres de l’acteur → action → prochain état latent → récompense → rendement
```

C’est indispensable dans DreamerV1 : l’acteur apprend en rétropropageant directement à travers la
dynamique imaginée. L’écrasement par `tanh` modifie la densité de probabilité ; sa correction jacobienne
est donc incluse dans `log_prob`, notamment pour calculer correctement la régularisation d’entropie.

### Le critique : estimer la valeur d’un état latent

Le critique $v_\psi(s_t)$ ne choisit aucune action. Il estime le rendement futur attendu depuis l’état
latent courant :

$$
v_\psi(s_t)
\approx
\mathbb E_{\pi_\phi}\!\left[
\sum_{k=0}^{\infty}\gamma^k\hat r_{t+k}
\,\middle|\,s_t
\right].
$$

Il sert à **prolonger** une imagination de longueur finie : lorsque le rollout latent s’arrête après
$H$ pas, le critique estime ce qui aurait probablement été obtenu ensuite. Il fournit ainsi le
*bootstrap* utilisé dans les $\lambda$-returns.

Les rôles sont complémentaires :

- l’**acteur** cherche les actions qui augmentent le rendement imaginé ;
- le **critique** apprend à prédire ce rendement et fournit une cible moins bruitée ;
- le **RSSM** relie les actions aux états futurs et permet au gradient de traverser le rêve.

Pour le détail de la gaussienne écrasée, de la correction du `tanh` et de la régularisation par entropie,
voir le notebook 08 consacré à SAC.

In [ ]:
class Actor(nn.Module):
    """Politique gaussienne écrasée par tanh sur la feature latente (rappel SAC, nb08)."""
    def __init__(self, feat_dim, action_dim, hidden_dim, low=-1.0, high=1.0):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(feat_dim, hidden_dim), nn.SiLU(),
                                   nn.Linear(hidden_dim, hidden_dim), nn.SiLU())
        self.mean = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Linear(hidden_dim, action_dim)
        self.register_buffer("scale", torch.tensor((high - low) / 2.0))
        self.register_buffer("bias",  torch.tensor((high + low) / 2.0))

    def sample(self, feat):
        h = self.trunk(feat)
        mean, log_std = self.mean(h), self.log_std(h).clamp(-5.0, 2.0)
        dist = torch.distributions.Normal(mean, log_std.exp())
        u = dist.rsample()                                  # reparamétrée -> gradient
        a = torch.tanh(u)
        logp = (dist.log_prob(u) - torch.log(1 - a.pow(2) + 1e-6)).sum(-1)
        return a * self.scale + self.bias, logp

    def act_mean(self, feat):
        """Action déterministe (greedy) : moyenne écrasée par tanh, sans échantillonnage."""
        h = self.trunk(feat)
        return torch.tanh(self.mean(h)) * self.scale + self.bias

class Critic(nn.Module):
    """Valeur d'état V(feat)."""
    def __init__(self, feat_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(feat_dim, hidden_dim), nn.SiLU(),
                                 nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
                                 nn.Linear(hidden_dim, 1))
    def forward(self, feat):
        return self.net(feat).squeeze(-1)

# Mini-test : action bornée, log-prob finie
_a, _lp = Actor(40, 6, 32).sample(torch.randn(4, 40))
assert _a.shape == (4, 6) and torch.isfinite(_lp).all() and _a.abs().max() <= 1.0 + 1e-5
print("Actor/Critic OK | a∈[-1,1] ✓ logπ fini ✓")

## III.3 — Les $\lambda$-returns

Pour évaluer une trajectoire imaginée, on n’utilise ni le seul pas suivant — **TD(0)**, peu bruité mais
potentiellement biaisé par les erreurs du critique — ni le rendement complet — **Monte-Carlo**, plus fidèle
aux récompenses mais de variance élevée. Dreamer combine les deux avec le **$\lambda$-return**, calculé
récursivement en remontant la trajectoire depuis son dernier état :

$$
V^\lambda_t
=
r_{t+1}
+
\gamma\Big[
(1-\lambda)\,v_\psi(s_{t+1})
+
\lambda\,V^\lambda_{t+1}
\Big],
\qquad
V^\lambda_H=v_\psi(s_H).
$$

Le terme $(1-\lambda)v_\psi(s_{t+1})$ fait confiance au critique dès l’état suivant, tandis que
$\lambda V^\lambda_{t+1}$ propage davantage les récompenses rencontrées plus loin dans l’imagination.
Le calcul à rebours mélange ainsi des rendements utilisant différentes profondeurs de bootstrap, de
1 jusqu’à $H$ pas.

- $\lambda=0$ donne **TD(0)** : bootstrap immédiat, faible variance mais forte dépendance au critique ;
- $\lambda=1$ se rapproche de **Monte-Carlo** : toutes les récompenses imaginées sont propagées avant le
  bootstrap final ;
- Dreamer utilise généralement $\lambda\approx0.95$, un compromis qui exploite presque tout le rollout
  imaginé tout en conservant la stabilisation fournie par le critique.

In [ ]:
def lambda_returns(rewards, values, gamma, lam):
    """rewards, values : [T, M] (T = H+1). Renvoie les λ-returns [T-1, M]."""
    T = rewards.shape[0]
    out = [None] * (T - 1)
    last = values[-1]
    for t in reversed(range(T - 1)):
        last = rewards[t + 1] + gamma * ((1 - lam) * values[t + 1] + lam * last)
        out[t] = last
    return torch.stack(out, dim=0)

# Mini-test : shape + finitude
_r = torch.randn(6, 10); _v = torch.randn(6, 10)
_ret = lambda_returns(_r, _v, 0.99, 0.95)
assert _ret.shape == (5, 10) and torch.isfinite(_ret).all()
print("λ-returns OK :", tuple(_ret.shape))

In [ ]:
# Visualisation : λ interpole TD(0) (biais) ↔ Monte-Carlo (variance)
gamma_demo, H_demo = 0.97, 25
# (a) biais : 1 trajectoire déterministe, valeur critique volontairement fausse (=8)
rew_det = torch.ones(H_demo + 1, 1)
val_bad = torch.full((H_demo + 1, 1), 8.0)
curves = {l: lambda_returns(rew_det, val_bad, gamma_demo, l)[:, 0].numpy() for l in [0.0, 0.5, 0.95, 1.0]}
# (b) variance : M trajectoires à récompense bruitée, valeur nulle
M_demo = 4000
rew_noisy = 1.0 + 0.5 * torch.randn(H_demo + 1, M_demo)
val_zero = torch.zeros(H_demo + 1, M_demo)
lams = np.linspace(0, 1, 11)
stds = [float(lambda_returns(rew_noisy, val_zero, gamma_demo, float(l))[0].std()) for l in lams]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for l in [0.0, 0.5, 0.95, 1.0]:
    ax[0].plot(curves[l], marker=".", label=f"λ={l}")
ax[0].axhline(8.0, color="gray", ls=":", label="valeur critique (biaisée)")
ax[0].set_title("Biais : λ interpole valeur ↔ récompenses"); ax[0].set_xlabel("pas t"); ax[0].set_ylabel("λ-return"); ax[0].legend(fontsize=8)
ax[1].plot(lams, stds, marker="o", color="crimson"); ax[1].set_title("Variance : écart-type du λ-return ↑ avec λ")
ax[1].set_xlabel("λ"); ax[1].set_ylabel("std du return")
fig.suptitle("λ-returns : compromis biais (λ→0, TD) / variance (λ→1, Monte-Carlo)")
fig.tight_layout(); plt.show()

**Lecture.** À gauche : pour $\lambda=0$ (TD(0)) le return colle à la **valeur critique** — s'il est biaisé, le return l'est aussi ; pour $\lambda=1$ (Monte-Carlo) il reflète les **récompenses** accumulées, ignorant la valeur. $\lambda=0.95$ se place entre les deux. À droite : quand les récompenses sont bruitées, l'**écart-type** du return **croît avec $\lambda$** — Monte-Carlo est non biaisé mais de forte variance. Dreamer choisit $\lambda\approx0.95$ : presque tout le signal des récompenses, un soupçon de stabilisation par la valeur.

## III.4 — Le gradient de valeur analytique : le cœur de DreamerV1

Voici l’idée centrale de DreamerV1. Une fois le modèle de monde appris, une trajectoire imaginée est un
grand **graphe de calcul différentiable** :

- l’acteur transforme l’état latent en action ;
- le RSSM transforme cette action en état latent suivant ;
- la tête de récompense évalue cet état ;
- le critique estime la valeur de la suite ;
- les $\lambda$-returns combinent ces quantités en un objectif final.

L’acteur ne reçoit donc pas seulement le message « cette trajectoire était bonne ». Il peut obtenir une
information beaucoup plus précise :

> « Si cette action latente avait légèrement changé, comment l’état suivant, la récompense et le rendement
> auraient-ils changé ? »

Grâce à la reparamétrisation de l’action et du latent stochastique, PyTorch peut appliquer la règle de la
chaîne à travers tout le rêve :

$$
\frac{\partial V^\lambda_t}{\partial\phi}
=
\frac{\partial V^\lambda_t}{\partial s_{t+1}}
\frac{\partial s_{t+1}}{\partial a_t}
\frac{\partial a_t}{\partial\phi}
+\text{contributions des pas suivants}.
$$

Chaque facteur a une interprétation :

- $\partial a_t/\partial\phi$ indique comment les paramètres de l’acteur modifient l’action ;
- $\partial s_{t+1}/\partial a_t$ décrit comment le modèle prévoit que cette action transforme le monde ;
- $\partial V^\lambda_t/\partial s_{t+1}$ indique dans quelle direction le futur imaginé devient plus
  avantageux ;
- les contributions suivantes propagent cet effet sur plusieurs pas du rollout.

Le terme **gradient analytique** ne signifie pas que l’on résout l’optimisation avec une formule fermée.
Il signifie que le gradient est obtenu directement par différentiation automatique du modèle appris,
plutôt que par différences finies ou uniquement à partir du score probabiliste de la politique.

### Perte de l’acteur

L’acteur cherche à maximiser les $\lambda$-returns imaginés. On minimise donc leur opposé, avec une petite
régularisation d’entropie :

$$
\mathcal L_{\mathrm{actor}}
=
-\mathbb E\!\left[V^\lambda_t\right]
-\eta\,\mathcal H\!\left(\pi_\phi(\cdot\mid s_t)\right).
$$

Le premier terme pousse l’acteur vers les actions qui produisent de meilleurs futurs dans le RSSM.
L’entropie évite que la politique ne devienne trop vite déterministe et conserve un peu de diversité dans
les actions imaginées.

### Perte du critique

Le critique apprend à prédire les mêmes $\lambda$-returns :

$$
\mathcal L_{\mathrm{critic}}
=
\frac12\,
\mathbb E\!\left[
\left(v_\psi(s_t)-\operatorname{stopgrad}(V^\lambda_t)\right)^2
\right].
$$

La cible est **détachée** avec `stopgrad` : lorsque le critique est mis à jour, le gradient ne doit pas
modifier en même temps le RSSM, l’acteur ou le calcul du return. Le critique poursuit ainsi une cible
temporairement fixe, ce qui stabilise la régression.

Le chemin du gradient de l’acteur traverse toute la dynamique imaginée :

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    phi["paramètres acteur $$~\phi$$"]
    a["action reparamétrée<br/>$$~a_t=\tanh(u_t)$$"]
    dyn["dynamique imaginée<br/>$$~s_{t+1}=g_\theta(s_t,a_t)$$"]
    r["récompense prédite<br/>$$~\hat r_{t+1}$$"]
    v["valeur prédite<br/>$$~v_\psi(s_{t+1})$$"]
    ret["rendement<br/>$$~V_t^\lambda$$"]
    loss["perte acteur<br/>$$~-\,\mathbb E[V_t^\lambda]$$"]

    phi --> a --> dyn
    dyn --> r --> ret
    dyn --> v --> ret
    ret --> loss
    loss -. "rétropropagation à travers le rêve" .-> phi
```

### Pourquoi est-ce différent de REINFORCE ([nb03](./03_reinforce_cartpole_walkthrough.ipynb)) ?

REINFORCE traite l’environnement comme une **boîte noire**. Il ne sait pas comment une petite variation
de l’action modifierait l’état suivant ; il utilise seulement le rendement observé pour pondérer
$\nabla_\phi\log\pi_\phi(a_t\mid s_t)$. Ce gradient de type *score function* fonctionne sans modèle
différentiable, mais il est généralement très bruité.

Dreamer possède au contraire une approximation différentiable du monde. Il peut exploiter les dérivées
de la dynamique et de la récompense pour obtenir un signal plus direct et plus riche. C’est le principe
des **value gradients through a learned model**.

Cette puissance comporte cependant un risque : si le modèle de monde est imparfait, l’acteur peut suivre
un gradient qui améliore le rendement **dans le rêve**, mais pas dans le monde réel. La qualité du RSSM et
la brièveté raisonnable des trajectoires imaginées sont donc essentielles.

In [ ]:
def imagine_and_losses(rssm, reward_model, actor, critic, start_state, horizon, gamma, lam, ent_coef):
    """Déroule `horizon` pas en imagination depuis start_state (détaché) ; renvoie (actor_loss, critic_loss, metrics)."""
    state = {k: v.detach() for k, v in start_state.items()}
    feats = [rssm.get_feat(state)]
    logps = []
    for _ in range(horizon):
        a, logp = actor.sample(rssm.get_feat(state))      # reparamétrée -> garde le graphe
        logps.append(logp)
        state = rssm.img_step(state, a)                   # dynamique différentiable
        feats.append(rssm.get_feat(state))
    feats = torch.stack(feats, dim=0)                     # [H+1, M, F]
    rewards = reward_model(feats)                         # [H+1, M]
    values = critic(feats)                                # [H+1, M]
    returns = lambda_returns(rewards, values, gamma, lam) # [H, M]

    entropy = -torch.stack(logps).mean()
    actor_loss = -returns.mean() - ent_coef * entropy     # gradient analytique à travers la dynamique
    critic_loss = 0.5 * ((critic(feats[:-1].detach()) - returns.detach()) ** 2).mean()
    return actor_loss, critic_loss, {"imagined_return": returns.mean().item()}

# Mini-test : pertes finies + le gradient atteint bien l'acteur
torch.manual_seed(0)
rssm2 = RSSM(6, 16, 32, 8, 32); rm2 = RewardModel(40, 32)
act2 = Actor(40, 6, 32); cri2 = Critic(40, 32)
start = rssm2.initial_state(12)
al, cl, mm = imagine_and_losses(rssm2, rm2, act2, cri2, start, horizon=5, gamma=0.99, lam=0.95, ent_coef=1e-4)
al.backward()
grad_norm = sum(p.grad.abs().sum() for p in act2.parameters() if p.grad is not None)
assert torch.isfinite(al) and torch.isfinite(cl) and grad_norm > 0
print(f"actor_loss={float(al):.3f} critic_loss={float(cl):.3f} | gradient atteint l'acteur ✓")

**Lecture.** L'assertion `grad_norm > 0` confirme l'essentiel : le gradient de la perte de l'acteur
**remonte jusqu'à ses paramètres en traversant la dynamique imaginée**. C'est ce chemin de gradient
analytique qui fait l'efficacité de DreamerV1. On a maintenant les deux moitiés : modèle de monde (II) et
apprentissage en imagination (III). Assemblons.

---
# Partie IV — L'algorithme complet

## IV.1 — Pseudocode de Dreamer

$$
\boxed{
\begin{aligned}
&\textbf{Init : } \text{encodeur, RSSM, décodeur, tête récompense, acteur, critique, buffer} \\
&\textbf{Warmup : } \text{collecter quelques épisodes (politique aléatoire)} \\
&\textbf{répéter :} \\
&\quad \text{(1) Collecte — à chaque pas réel :} \\
&\quad\quad e_t = \mathrm{enc}(o_t);\quad s_t \leftarrow \text{posterior}(s_{t-1}, a_{t-1}, e_t);\quad a_t \sim \pi_\phi(\cdot \mid s_t) \\
&\quad\quad o_{t+1}, r_t \leftarrow \text{env}(a_t);\quad \text{stocker } (o_t, a_t, r_t) \\
&\quad \text{(2) Modèle de monde : séquences } (B \times L),\ \min\ \mathrm{recon} + \mathrm{reward} + \beta\,\mathrm{KL} \\
&\quad \text{(3) Comportement : depuis les posteriors (détachés), imaginer } H \text{ pas} \\
&\quad\quad \text{calculer les } \lambda\text{-returns} \Rightarrow \text{critique (régression)},\ \text{acteur (gradient analytique)} \\
\end{aligned}
}
$$

La même boucle, vue d'ensemble (le réel ne sert qu'à la collecte ; modèle et politique apprennent hors-ligne) :

```mermaid
flowchart TB
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    env["Environnement réel"]
    subgraph loop["Boucle Dreamer"]
        collect["1. COLLECTER (posterior)<br/>$$~a_t \sim \pi_\phi(\cdot \mid s_t)$$"]
        world["2. MODÈLE DE MONDE<br/>$$~\mathrm{recon} + \mathrm{reward} + \beta\,\mathrm{KL}$$"]
        behav["3. COMPORTEMENT (imagination)<br/>$$~H \text{ pas rêvés, } \lambda\text{-returns}$$"]
    end
    env -->|"observations réelles"| collect
    collect -->|"séquences → buffer"| world
    world -->|"posteriors = états de départ"| behav
    behav -->|"politique améliorée"| collect
    collect -->|"agit"| env
```

## IV.2 — L'environnement : HalfCheetah-v5

Un guépard planaire à 6 articulations. **Observation** (17-dim) : angles et positions du tronc et des
articulations (≈ 8 dims) + leurs vitesses (≈ 9 dims) — la position horizontale $x$ est **exclue** (non
stationnaire). **Action** (6-dim ∈ $[-1,1]$) : les couples appliqués aux 6 articulations. **Récompense** :
vitesse d'avancement − petit coût de contrôle. Pas de terminaison anticipée (l'épisode est seulement
*tronqué* dans le temps).

In [ ]:
env = gym.make("HalfCheetah-v5")
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]
act_low  = torch.tensor(env.action_space.low,  dtype=torch.float32)
act_high = torch.tensor(env.action_space.high, dtype=torch.float32)
print(f"obs_dim={obs_dim}  act_dim={act_dim}  action ∈ [{float(act_low[0])}, {float(act_high[0])}]")
o0, _ = env.reset(seed=0)
print("exemple d'observation (5 premières dims) :", np.round(o0[:5], 3))


## IV.3 — Le buffer de séquences

Le RSSM s'entraîne sur des **séquences contiguës** (pour apprendre la dynamique temporelle), pas sur des
transitions i.i.d. On écrit donc un buffer qui **stocke des épisodes** et **échantillonne des fenêtres**
de longueur $L$.

In [ ]:
class SequenceBuffer:
    """Stocke des épisodes complets ; échantillonne des fenêtres contiguës de longueur L."""
    def __init__(self, capacity):
        self.capacity = capacity
        self.episodes = []
        self.current = []

    def add(self, obs, action, reward, done):
        self.current.append((np.asarray(obs, np.float32), np.asarray(action, np.float32),
                             float(reward), bool(done)))
        if done:
            self.flush()

    def flush(self):
        if not self.current:
            return
        o, a, r, d = zip(*self.current)
        self.episodes.append({"obs": np.stack(o), "action": np.stack(a),
                              "reward": np.array(r, np.float32), "done": np.array(d, np.float32)})
        self.current = []
        while sum(e["obs"].shape[0] for e in self.episodes) > self.capacity and len(self.episodes) > 1:
            self.episodes.pop(0)

    def sample(self, batch_size, seq_len):
        elig = [e for e in self.episodes if e["obs"].shape[0] >= seq_len]
        if not elig:
            raise RuntimeError(f"aucun épisode de longueur >= {seq_len}")
        ob, ac, re = [], [], []
        for _ in range(batch_size):
            e = elig[np.random.randint(len(elig))]
            s = np.random.randint(0, e["obs"].shape[0] - seq_len + 1)
            sl = slice(s, s + seq_len)
            ob.append(e["obs"][sl]); ac.append(e["action"][sl]); re.append(e["reward"][sl])
        return (torch.tensor(np.stack(ob)), torch.tensor(np.stack(ac)), torch.tensor(np.stack(re)))

    def __len__(self):
        return sum(e["obs"].shape[0] for e in self.episodes) + len(self.current)

# Mini-test
_b = SequenceBuffer(1000)
for ep in range(3):
    for k in range(30):
        _b.add(np.zeros(17), np.zeros(6), 0.1, done=(k == 29))
_ob, _ac, _re = _b.sample(4, 10)
assert _ob.shape == (4, 10, 17) and _ac.shape == (4, 10, 6) and _re.shape == (4, 10)
print("SequenceBuffer OK :", tuple(_ob.shape))

## IV.4 — Assembler l'agent et la boucle d'entraînement

On instancie tous les modules (**petits** : pour tenir en quelques minutes sur CPU). On garde un état
récurrent pendant l'acte, on **scelle les épisodes** assez court (`MAX_STEPS` petit) pour que le buffer ait
de quoi échantillonner rapidement, et on alterne update du modèle de monde et update du comportement.

In [ ]:
class DreamerAgent:
    def __init__(self, obs_dim, act_dim, deter, stoch, hidden, embed, buffer_capacity=50_000):
        feat_dim = deter + stoch
        self.act_dim = act_dim
        self.encoder = Encoder(obs_dim, hidden, embed)
        self.rssm = RSSM(act_dim, embed, deter, stoch, hidden)
        self.decoder = Decoder(feat_dim, hidden, obs_dim)
        self.reward_model = RewardModel(feat_dim, hidden)
        self.actor = Actor(feat_dim, act_dim, hidden, low=-1.0, high=1.0)
        self.critic = Critic(feat_dim, hidden)
        self.buffer = SequenceBuffer(buffer_capacity)

        self.model_params = (
            list(self.encoder.parameters()) + list(self.rssm.parameters())
            + list(self.decoder.parameters()) + list(self.reward_model.parameters())
        )
        self.model_optimizer = torch.optim.Adam(self.model_params, lr=6e-4)
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=8e-5)
        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=8e-5)
        self.reset_episode_state()

    def reset_episode_state(self):
        self._state = None
        self._previous_action = None

    @torch.no_grad()
    def select_action(self, observation, warmup_steps=0, deterministic=False):
        if len(self.buffer) < warmup_steps:
            return np.random.uniform(-1.0, 1.0, size=self.act_dim).astype(np.float32)

        if self._state is None:
            self._state = self.rssm.initial_state(1)
            self._previous_action = torch.zeros(1, self.act_dim)

        embedding = self.encoder(torch.as_tensor(observation, dtype=torch.float32)[None])
        posterior, _ = self.rssm.obs_step(self._state, self._previous_action, embedding)
        feature = self.rssm.get_feat(posterior)

        action = self.actor.act_mean(feature) if deterministic else self.actor.sample(feature)[0]
        self._state, self._previous_action = posterior, action
        return action.squeeze(0).numpy().astype(np.float32)

    def store_transition(self, observation, action, reward, done):
        self.buffer.add(observation, action, reward, done=done)

    def end_episode(self):
        self.buffer.flush()
        self.reset_episode_state()

    def update_world_model(self, batch_size, sequence_length, free_nats):
        obs, action, reward = self.buffer.sample(batch_size, sequence_length)
        batch, length, _ = obs.shape

        embed = self.encoder(obs)

        # feature_t est construit avec prev_action = a_{t-1}, puis observation o_t.
        previous_actions = torch.cat([torch.zeros_like(action[:, :1]), action[:, :-1]], dim=1)

        state = self.rssm.initial_state(batch)
        features, kls, deters, stochs = [], [], [], []

        for step in range(length):
            posterior, prior = self.rssm.obs_step(state, previous_actions[:, step], embed[:, step])
            features.append(self.rssm.get_feat(posterior))
            kls.append(self.rssm.kl(posterior, prior, free_nats))
            deters.append(posterior["deter"])
            stochs.append(posterior["stoch"])
            state = posterior

        feature = torch.stack(features, dim=1)  # [B, L, F]

        # Reconstruction : feature_t doit reconstruire obs_t.
        reconstruction = 0.5 * ((self.decoder(feature) - obs) ** 2).sum(-1).mean()

        # Reward : reward[:, t] vient de l'action a_t, donc correspond au latent suivant feature_{t+1}.
        # On entraîne donc feature[:, 1:] -> reward[:, :-1].
        reward_pred = self.reward_model(feature[:, 1:])
        reward_target = reward[:, :-1]
        reward_loss = 0.5 * ((reward_pred - reward_target) ** 2).mean()

        kl = torch.stack(kls).mean()
        loss = reconstruction + reward_loss + kl

        self.model_optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.model_params, 100.0)
        self.model_optimizer.step()

        start = {
            "deter": torch.stack(deters, 1).reshape(batch * length, -1).detach(),
            "stoch": torch.stack(stochs, 1).reshape(batch * length, -1).detach(),
        }
        return reconstruction.item(), reward_loss.item(), kl.item(), start

    def update_behavior(self, start, horizon, gamma, lam, entropy_coef):
        actor_loss, critic_loss, metrics = imagine_and_losses(
            self.rssm, self.reward_model, self.actor, self.critic,
            start, horizon, gamma, lam, entropy_coef,
        )

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        nn.utils.clip_grad_norm_(self.actor.parameters(), 100.0)
        self.actor_optimizer.step()

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), 100.0)
        self.critic_optimizer.step()

        return actor_loss.item(), critic_loss.item(), metrics["imagined_return"]

In [ ]:
# ---- hyperparamètres Dreamer un peu plus solides ----
DETER, STOCH, HID, EMB = 128, 32, 128, 64
FEAT = DETER + STOCH

HORIZON, GAMMA, LAM, FREE_NATS, ENT = 15, 0.99, 0.95, 1.0, 1e-4
BATCH, SEQLEN = 32, 32

MAX_STEPS = 300
WARMUP = 5_000
TOTAL_STEPS = 50_000
TRAIN_EVERY = 1
EVAL_EVERY, EVAL_EPISODES = 5_000, 3

torch.manual_seed(0)
np.random.seed(0)

agent = DreamerAgent(obs_dim, act_dim, DETER, STOCH, HID, EMB, buffer_capacity=100_000)

encoder, rssm = agent.encoder, agent.rssm
decoder, reward_model = agent.decoder, agent.reward_model
actor, critic, buffer = agent.actor, agent.critic, agent.buffer

print("DreamerAgent assemblé : monde, comportement, buffer et mémoire récurrente")

In [ ]:
@torch.no_grad()
def rollout_eval(seed, n_steps=300, deterministic=True, return_reward=False):
    e = gym.make("HalfCheetah-v5")
    o, _ = e.reset(seed=seed)
    x0 = float(e.unwrapped.data.qpos[0])

    s = agent.rssm.initial_state(1)
    prev_a = torch.zeros(1, act_dim)
    total_reward = 0.0

    for _ in range(n_steps):
        emb = agent.encoder(torch.tensor(o, dtype=torch.float32)[None])
        post, _ = agent.rssm.obs_step(s, prev_a, emb)
        feat = agent.rssm.get_feat(post)
        a = agent.actor.act_mean(feat) if deterministic else agent.actor.sample(feat)[0]
        a_np = a.squeeze(0).numpy().astype(np.float32)

        o, r, term, trunc, _ = e.step(a_np)
        total_reward += float(r)

        s, prev_a = post, a
        if term or trunc:
            break

    dx = float(e.unwrapped.data.qpos[0]) - x0
    e.close()

    if return_reward:
        return dx, total_reward
    return dx

print("rollout_eval défini : Δx + reward optionnelle")

In [ ]:
# ---- boucle d'entraînement : warmup aléatoire, puis collecte + apprentissage ----
hist = {
    "recon": [],
    "reward": [],
    "kl": [],
    "actor": [],
    "critic": [],
    "imagined_return": [],
    "ep_return": [],
    "eval_dx": [],
}
obs, _ = env.reset(seed=0)
ep_ret, ep_len = 0.0, 0
last_eval_dx = float("nan")

pbar = tqdm(
    range(TOTAL_STEPS),
    desc="Dreamer",
    unit="pas réel",
    mininterval=0.5,
    dynamic_ncols=True,
)

for step in pbar:
    # 1. Agir dans le monde réel et stocker la transition.
    action = agent.select_action(obs, warmup_steps=WARMUP)
    next_obs, reward, terminated, truncated, _ = env.step(action)
    agent.store_transition(obs, action, reward, done=bool(terminated))

    ep_ret += reward
    ep_len += 1
    obs = next_obs

    # 2. Fermer l'épisode et réinitialiser la mémoire récurrente.
    done_ep = terminated or truncated or ep_len >= MAX_STEPS
    if done_ep:
        agent.end_episode()
        hist["ep_return"].append(ep_ret)

        obs, _ = env.reset(seed=step)
        ep_ret, ep_len = 0.0, 0

    # 3. Apprendre le modèle de monde, puis le comportement en imagination.
    ready_to_train = len(buffer) >= max(WARMUP, BATCH * SEQLEN)
    if ready_to_train and step % TRAIN_EVERY == 0:
        rc, rw, kl, start = agent.update_world_model(BATCH, SEQLEN, FREE_NATS)
        al, cl, imagined_return = agent.update_behavior(start, HORIZON, GAMMA, LAM, ENT)

        metrics = {
            "recon": rc,
            "reward": rw,
            "kl": kl,
            "actor": al,
            "critic": cl,
            "imagined_return": imagined_return,
        }
        for name, value in metrics.items():
            hist[name].append(value)

    # 4. Évaluer périodiquement la politique greedy dans des environnements neufs.
    if step % EVAL_EVERY == 0 and len(buffer) >= WARMUP:
        eval_distances = [
            rollout_eval(7000 + j, n_steps=300)
            for j in range(EVAL_EPISODES)
        ]
        last_eval_dx = float(np.mean(eval_distances))
        hist["eval_dx"].append((step, last_eval_dx))

    # 5. Afficher quelques diagnostics sans ralentir la boucle à chaque pas.
    if step % 100 == 0 or done_ep or step == TOTAL_STEPS - 1:
        phase = "warmup" if len(buffer) < WARMUP else "train"

        postfix = {
            "phase": phase,
            "buffer": len(buffer),
            "episodes": len(hist["ep_return"]),
        }

        if hist["ep_return"]:
            postfix["ret_ep"] = f"{hist['ep_return'][-1]:.1f}"
        if hist["recon"]:
            postfix["recon"] = f"{hist['recon'][-1]:.3f}"
            postfix["KL"] = f"{hist['kl'][-1]:.3f}"
            postfix["Vλ"] = f"{hist['imagined_return'][-1]:.2f}"
        if np.isfinite(last_eval_dx):
            postfix["eval_Δx"] = f"{last_eval_dx:.2f}"

        pbar.set_postfix(postfix)

pbar.close()

print(
    f"Terminé : {len(hist['recon'])} updates du modèle, "
    f"{len(hist['ep_return'])} épisodes réels et "
    f"{len(hist['eval_dx'])} évaluations."
)


---
# Partie V — Diagnostics d'entraînement

## V.1 — Les courbes de pertes

On regarde si chaque morceau apprend : reconstruction et récompense doivent **baisser** ; le KL doit se
**stabiliser** (au-dessus des free-nats) ; acteur/critique sont **bruités** à cette micro-échelle.

In [ ]:
def smooth(x, k=15):
    x = np.asarray(x, dtype=np.float32)
    return np.convolve(x, np.ones(k) / k, mode="valid") if len(x) >= k else x

fig, ax = plt.subplots(2, 3, figsize=(13, 6))
panels = [("recon", "reconstruction"), ("reward", "récompense"), ("kl", "KL"),
          ("actor", "perte acteur"), ("critic", "perte critique"), ("imagined_return", "rendement imaginé")]
for a, (key, title) in zip(ax.ravel(), panels):
    a.plot(smooth(hist[key])); a.set_title(title); a.set_xlabel("update")
fig.suptitle("Diagnostics Dreamer (micro-échelle)"); fig.tight_layout(); plt.show()

In [ ]:
# Éval greedy périodique : distance avant (Δx) de la politique au fil de l'entraînement
if hist["eval_dx"]:
    ev = np.array(hist["eval_dx"], dtype=float)
    plt.figure(figsize=(7, 3.2))
    plt.plot(ev[:, 0], ev[:, 1], marker="o", color="#3b7")
    plt.axhline(0, color="k", lw=0.8)
    plt.xlabel("pas réels"); plt.ylabel("distance avant Δx (greedy)")
    plt.title("Politique greedy : avancement vs pas d'entraînement")
    plt.show()
else:
    print("Pas d'éval enregistrée (budget trop court).")

**Lecture.** Cette courbe mesure, à intervalles réguliers, la **distance avant** parcourue par la politique **greedy** (action déterministe) dans un environnement neuf — un signal de comportement plus direct que les pertes. On veut la voir **monter au-dessus de 0** (mieux que rester sur place) au fil de l'entraînement ; à cette échelle le signal reste modeste mais réel.

**Lecture.** La perte de **reconstruction** descend nettement : l'encodeur/décodeur capturent les 17
dimensions. La perte de **récompense** descend aussi : la tête récompense apprend à prédire la vitesse
d'avancement. Le **KL** se stabilise (le prior a appris à imiter le posterior — l'imagination devient
utilisable). Le **rendement imaginé** monte (l'acteur apprend à exploiter le modèle). Les courbes
**acteur/critique** sont bruitées : à cette échelle minuscule (quelques milliers d'updates), on montre la
*mécanique*, pas la maîtrise — exactement comme pour MBPO (nb12).

---
# Partie VI — Démonstrations : que sait faire le modèle de monde ?

## VI.1 — (a) Reconstruction : le décodeur rejoue les 17 dimensions

On prend un vrai épisode, on encode chaque observation, on déroule le **posterior** (donc en *regardant*
les observations), et on **décode**. Si le modèle de monde est bon, les 17 dimensions reconstruites suivent
les vraies.

In [ ]:
@torch.no_grad()
def rollout_real_episode(n_steps=MAX_STEPS, seed=123):
    agent.reset_episode_state()
    o, _ = env.reset(seed=seed); O, Acts = [o.astype(np.float32)], []
    for _ in range(n_steps):
        a = agent.select_action(o, deterministic=True)
        o, r, term, trunc, _ = env.step(a)
        O.append(o.astype(np.float32)); Acts.append(a.astype(np.float32))
        if term or trunc: break
    return np.stack(O[:-1]), np.stack(Acts)

real_obs, real_act = rollout_real_episode()
T = real_obs.shape[0]

@torch.no_grad()
def reconstruct(real_obs, real_act):
    obs = torch.tensor(real_obs)[None]; action = torch.tensor(real_act)[None]
    embed = encoder(obs); prev_a = torch.cat([torch.zeros_like(action[:, :1]), action[:, :-1]], 1)
    state = rssm.initial_state(1); feats = []
    for t in range(obs.shape[1]):
        post, _ = rssm.obs_step(state, prev_a[:, t], embed[:, t]); feats.append(rssm.get_feat(post)); state = post
    return decoder(torch.stack(feats, 1))[0].numpy()

recon_obs = reconstruct(real_obs, real_act)
dims = [0, 1, 8, 9]
fig, ax = plt.subplots(1, len(dims), figsize=(14, 3.2))
for a, d in zip(ax, dims):
    a.plot(real_obs[:, d], "k", label="réel"); a.plot(recon_obs[:, d], "--", label="reconstruit")
    a.set_title(f"dim {d}"); a.set_xlabel("pas")
ax[0].legend(); fig.suptitle("Reconstruction par le décodeur (posterior, avec observations)")
fig.tight_layout(); plt.show()

**Lecture.** En mode posterior (le modèle *voit* chaque observation), le décodeur reproduit fidèlement les
dimensions choisies : l'information des 17 dim **transite bien** par l'état latent compact. C'est la
validation de l'axe encodeur→latent→décodeur.

## VI.2 — (b) Le rêve en boucle ouverte : prédire les 17 dim à partir des seules actions

Le vrai test du modèle de monde : on le conditionne sur les **$K$ premières** observations (posterior pour
« s'amorcer »), puis on **coupe les observations** et on **imagine** la suite avec `img_step` — en ne lui
donnant **que les actions**. Le décodeur prédit alors les 17 dimensions *uniquement à partir des actions*.
C'est exactement « prédire l'état futur en fonction de nos actions ».

In [ ]:
@torch.no_grad()
def open_loop_dream(real_obs, real_act, warmup_k=5):
    obs = torch.tensor(real_obs)[None]; action = torch.tensor(real_act)[None]
    embed = encoder(obs); prev_a = torch.cat([torch.zeros_like(action[:, :1]), action[:, :-1]], 1)
    state = rssm.initial_state(1); preds = []
    Tn = obs.shape[1]
    for t in range(Tn):
        if t < warmup_k:
            state, _ = rssm.obs_step(state, prev_a[:, t], embed[:, t])   # amorçage : avec observation
        else:
            state = rssm.img_step(state, prev_a[:, t])                    # rêve : actions seules
        preds.append(decoder(rssm.get_feat(state))[0])
    return torch.stack(preds).numpy()

dream_obs = open_loop_dream(real_obs, real_act, warmup_k=5)

fig, ax = plt.subplots(1, len(dims), figsize=(14, 3.2))
for a, d in zip(ax, dims):
    a.plot(real_obs[:, d], "k", label="réel"); a.plot(dream_obs[:, d], "--", label="rêvé (actions seules)")
    a.axvline(5, color="r", ls=":", lw=1, label="fin amorçage")
    a.set_title(f"dim {d}"); a.set_xlabel("pas")
ax[0].legend(fontsize=8); fig.suptitle("Rêve en boucle ouverte : prédire les 17 dim à partir des actions")
fig.tight_layout(); plt.show()

# erreur de prédiction en fonction de l'horizon (au-delà de l'amorçage)
err = np.sqrt(((dream_obs - real_obs) ** 2).mean(axis=1))
plt.figure(figsize=(7, 3.2))
plt.plot(err, marker="."); plt.axvline(5, color="r", ls=":", label="fin amorçage")
plt.xlabel("pas (horizon)"); plt.ylabel("RMSE sur les 17 dim"); plt.title("L'erreur du rêve croît avec l'horizon")
plt.legend(); plt.show()

**Lecture.** Pendant l'amorçage (avant la ligne rouge), le modèle suit le réel. Après, **sans aucune
observation**, il continue à prédire les 17 dimensions à partir des seules actions : il **rêve** la
trajectoire. L'erreur **croît avec l'horizon** — l'erreur du modèle se *compose*, exactement comme dans
PETS/MBPO. C'est *pourquoi* Dreamer apprend sur des rollouts imaginés **courts** branchés sur des états
réels, plutôt que sur de longs rêves qui divergent.

## VI.3 — (c) Signal de politique apprise dans l'imagination

L'acteur n'a **jamais** été entraîné sur l'environnement réel — seulement dans le rêve. Mesurons un signal
**modeste** : la distance parcourue par la politique apprise vs une politique aléatoire (moyenne sur
quelques épisodes courts).

In [ ]:
# Éval finale : la politique greedy (apprise UNIQUEMENT en imagination) vs le hasard
@torch.no_grad()
def random_forward(seed, n_steps=300):
    e = gym.make("HalfCheetah-v5"); e.reset(seed=seed)
    x0 = float(e.unwrapped.data.qpos[0])
    for _ in range(n_steps):
        _, _, term, trunc, _ = e.step(np.random.uniform(-1, 1, size=act_dim).astype(np.float32))
        if term or trunc:
            break
    dx = float(e.unwrapped.data.qpos[0]) - x0; e.close()
    return dx

N_EVAL = 5
dre = [rollout_eval(500 + i, n_steps=300, deterministic=True) for i in range(N_EVAL)]
ran = [random_forward(500 + i, n_steps=300) for i in range(N_EVAL)]
dm, ds = float(np.mean(dre)), float(np.std(dre))
rm_, rs = float(np.mean(ran)), float(np.std(ran))
plt.figure(figsize=(5.5, 3.4))
plt.bar(["Dreamer\n(rêvé)", "aléatoire"], [dm, rm_], yerr=[ds, rs], capsize=6, color=["#3b7", "#bbb"])
plt.ylabel("distance avant (Δx)"); plt.title("Politique apprise UNIQUEMENT dans l'imagination")
plt.axhline(0, color="k", lw=0.8); plt.show()
print(f"Dreamer Δx = {dm:.2f} ± {ds:.2f}   |   aléatoire Δx = {rm_:.2f} ± {rs:.2f}")

In [ ]:
# Démonstration finale : replay vidéo notebook, sans fenêtre une fenêtre locale.

In [ ]:
@torch.no_grad()
def evaluate_and_display_agent(label="Dreamer", seed=12_345, n_steps=300, video_root="videos/13_dreamer"):
    """Évalue la politique Dreamer greedy et affiche un replay vidéo enregistré."""
    safe_label = label.lower().replace(" ", "_").replace("-", "_")
    video_dir = Path(video_root) / safe_label
    video_dir.mkdir(parents=True, exist_ok=True)

    env_video = gym.make("HalfCheetah-v5", render_mode="rgb_array")
    env_video = RecordVideo(
        env_video,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == 0,
        name_prefix=f"{safe_label}_greedy",
    )

    obs, _ = env_video.reset(seed=seed)
    x0 = float(env_video.unwrapped.data.qpos[0])
    state = agent.rssm.initial_state(1)
    prev_action = torch.zeros(1, act_dim)
    total_reward = 0.0
    steps = 0

    try:
        for _ in range(n_steps):
            emb = agent.encoder(torch.tensor(obs, dtype=torch.float32)[None])
            post, _ = agent.rssm.obs_step(state, prev_action, emb)
            feat = agent.rssm.get_feat(post)
            action = agent.actor.act_mean(feat)
            action_np = action.squeeze(0).numpy().astype(np.float32)
            obs, reward, terminated, truncated, _ = env_video.step(action_np)
            total_reward += float(reward)
            state, prev_action = post, action
            steps += 1
            if terminated or truncated:
                break
    finally:
        dx = float(env_video.unwrapped.data.qpos[0]) - x0
        env_video.close()

    print(f"Démo {label} | Δx={dx:+.2f} | retour={total_reward:+.1f} | durée={steps} pas")

    videos = sorted(video_dir.glob("*.mp4"), key=lambda path: path.stat().st_mtime)
    if videos:
        print(f"Replay vidéo {label} :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=640))
    else:
        print(f"Aucune vidéo générée pour {label}.")

    return dx, total_reward


dreamer_demo_dx, dreamer_demo_return = evaluate_and_display_agent()


**Lecture (honnête).** La politique — entraînée *exclusivement dans l'imagination* du modèle de monde, sans
jamais optimiser sur l'environnement réel — avance **davantage que le hasard** sur HalfCheetah. À cette
micro-échelle (modèle minuscule, quelques milliers d'updates), on ne vise pas la performance d'un Dreamer
entraîné des heures ; le signal est **modeste mais réel** : la boucle « encoder → rêver → apprendre la
politique » fonctionne. La preuve la plus fiable reste la courbe de **rendement imaginé** (Partie V), que
l'acteur fait monter par construction. Sur HalfCheetah vectoriel (sans pixels), le gain de l'encodeur reste
ténu (cf. §2) ; le point du notebook est la **compréhension du pipeline**, pas le record.

---
# Partie VII — Conclusion, limites et suite

## VII.1 — Ce qu'on a construit

Dans ce notebook, on a construit une version pédagogique de **DreamerV1** : un agent model-based qui
n'apprend plus directement depuis le vrai environnement, mais depuis un **monde latent** appris.

La boucle complète est :

| Brique | Rôle |
|--------|------|
| **Encodeur** | compresse l'observation $o_t$ en embedding $e_t$ |
| **RSSM** | maintient un état latent $s_t=[h_t,z_t]$ avec mémoire déterministe et variable stochastique |
| **Prior / posterior** | prédit sans observation puis corrige avec l'observation, comme un filtre de Kalman neural |
| **Décodeur** | force le latent à contenir assez d'information pour reconstruire l'observation |
| **Tête récompense** | apprend à scorer les trajectoires imaginées |
| **Acteur / critique** | apprennent uniquement dans l'imagination, sur des rollouts latents |
| **$\lambda$-returns** | construisent des cibles de valeur entre TD(0) et Monte-Carlo |
| **Gradient de valeur analytique** | rétropropage le rendement imaginé à travers la dynamique jusqu'à l'acteur |

Le saut conceptuel par rapport à MBPO est majeur. MBPO imagine de **courts rollouts dans l'espace
d'état** et y entraîne SAC. Dreamer apprend d'abord un **espace latent** où il peut imaginer plus
longtemps, puis entraîne directement l'acteur et le critique dans cet espace.

$$
\underbrace{\text{MBPO}}_{\substack{\text{rollouts courts} \\ \text{dans l'état brut}}}
\quad\longrightarrow\quad
\underbrace{\text{Dreamer}}_{\substack{\text{rollouts latents} \\ \text{acteur entraîné dans le rêve}}}
$$

## VII.2 — Ce que Dreamer apporte

Dreamer est important parce qu'il sépare trois problèmes qui étaient mélangés dans les méthodes
précédentes :

1. **Comprendre le monde** : apprendre un modèle latent qui prédit observations et récompenses.
2. **Imaginer des futurs** : dérouler ce modèle sans toucher à l'environnement réel.
3. **Améliorer le comportement** : optimiser acteur et critique sur ces trajectoires imaginées.

L'idée centrale est donc :

$$
\text{expérience réelle}
\;\to\;
\text{modèle latent}
\;\to\;
\text{imagination}
\;\to\;
\text{politique améliorée}.
$$

Ce qui rend Dreamer différent d'un simple modèle de dynamique, c'est que le latent n'est pas seulement
un outil de prédiction. Il devient le **monde de travail** de l'agent : la valeur, la politique, les
récompenses et les gradients vivent tous dans cet espace.

C'est aussi pourquoi le gradient de valeur analytique est si puissant. Dans REINFORCE ou SAC, le vrai
environnement reste une boîte noire. Dans Dreamer, le futur imaginé est différentiable : l'acteur reçoit
un signal qui traverse la dynamique, la récompense et la valeur.

## VII.3 — Limites de notre version

Cette version reste volontairement petite et pédagogique. Elle montre la mécanique, pas l'état de l'art.

| Limite | Conséquence |
|--------|-------------|
| **Biais du modèle latent** | si le RSSM se trompe, l'acteur peut exploiter des hallucinations |
| **Erreur composée** | en boucle ouverte, les petites erreurs de dynamique s'accumulent avec l'horizon |
| **Updates coûteuses** | chaque update demande un déroulé séquentiel du RSSM puis une imagination de $H$ pas |
| **Gain latent modeste sur vecteurs** | HalfCheetah fournit déjà un état compact ; le vrai intérêt explose sur pixels |
| **Pas de tête continuation** | adapté ici à HalfCheetah, mais insuffisant pour des environnements avec mort terminale |
| **Micro-budget** | les courbes illustrent le mécanisme ; elles ne remplacent pas un entraînement DMC/Atari complet |

Le point le plus important est le premier : **l'acteur apprend dans le rêve**. Si le rêve est faux,
l'amélioration de politique peut être fausse aussi. Les diagnostics de reconstruction, de reward et de
rollout ouvert ne sont donc pas décoratifs : ils disent si l'imagination mérite d'être crue.

## VII.4 — Pourquoi la tête continuation manque ici

Dans notre HalfCheetah simplifié, l'épisode ne se termine pas vraiment par une mort dynamique ; il est
surtout tronqué par l'horizon. On peut donc imaginer $H$ pas sans apprendre explicitement une probabilité
de continuation.

Mais dans un environnement terminant, par exemple un robot qui tombe ou un jeu avec état de mort, ce
n'est plus correct. Il faut apprendre :

$$
\hat\gamma_t = p(\text{continue} \mid s_t)
$$

et utiliser cette continuation pour couper les rendements imaginés après la mort. DreamerV2/V3 rendent
cette idée centrale : le modèle ne prédit pas seulement récompense et observation, il prédit aussi si le
futur continue.

## VII.5 — Au-delà : DreamerV2 et DreamerV3

Nous avons construit l'esprit de **DreamerV1** : RSSM à latents gaussiens continus, imagination, acteur
et critique entraînés par $\lambda$-returns et gradients à travers le modèle.

Les versions suivantes gardent le même squelette, mais le rendent beaucoup plus robuste.

| Version | Apport principal |
|---------|------------------|
| **DreamerV1** | acteur/critique entraînés dans l'imagination latente |
| **DreamerV2** | latents discrets catégoriels + KL balancing, très fort sur Atari |
| **DreamerV3** | normalisations, symlog, two-hot, free bits : mêmes hyperparamètres sur de nombreux domaines |

**DreamerV2** remplace notamment les latents gaussiens par des variables discrètes catégorielles,
souvent plus stables et expressives pour des mondes riches. Il introduit aussi le **KL balancing** :
on ne pousse pas prior et posterior de façon naïvement symétrique, on aide le prior à apprendre à
prédire sans écraser la représentation.

**DreamerV3** pousse l'idée de robustesse : valeurs et récompenses transformées par `symlog`, cibles
two-hot, normalisations plus systématiques, et un seul jeu d'hyperparamètres sur des domaines très
différents. Le but n'est plus seulement de faire marcher Dreamer sur une tâche, mais de le rendre
fiable presque partout.

## VII.6 — Pont vers les modèles monde modernes

Dreamer appartient à une famille plus large de **world models** : on apprend un espace interne où
prédire et agir devient plus facile que dans l'observation brute.

La lignée du notebook se lit ainsi :

$$
\underbrace{\text{PETS}}_{\text{modèle d'état + planning}}
\to
\underbrace{\text{MBPO}}_{\text{modèle d'état + politique SAC}}
\to
\underbrace{\text{Dreamer}}_{\text{modèle latent + politique dans l'imagination}}
\to
\underbrace{\text{MuZero / JEPA}}_{\text{latents décisionnels ou prédictifs}}
$$

Dreamer garde une contrainte générative : il reconstruit encore l'observation. MuZero, lui, ne demande
pas au latent de reconstruire le monde, seulement de soutenir recherche, récompense et valeur. JEPA va
encore déplacer la question : apprendre des représentations prédictives sans reconstruction pixel par
pixel. Ces différences deviennent centrales dans les notebooks suivants.

## VII.7 — Ce que vous devriez pouvoir réexpliquer

À la fin de ce chapitre, vous devriez pouvoir répondre à ces questions :

1. Pourquoi un SSM sépare état latent, entrée et sortie.
2. Pourquoi la discrétisation transforme un système continu en récurrence utilisable.
3. Pourquoi le RSSM combine une mémoire déterministe $h_t$ et un latent stochastique $z_t$.
4. Pourquoi le prior prédit sans observation, alors que le posterior corrige avec l'observation.
5. Comment l'ELBO combine reconstruction, récompense et KL.
6. Pourquoi les free-nats évitent de rendre $z_t$ inutile.
7. Comment les $\lambda$-returns mélangent bootstrap et Monte-Carlo.
8. Pourquoi le gradient de l'acteur peut traverser la dynamique imaginée.
9. Pourquoi Dreamer est plus naturel que MBPO dès que les observations deviennent riches.
10. Pourquoi DreamerV2/V3 ajoutent continuation, latents discrets, symlog et two-hot.

> **Lien avec le code du dépôt.** Une version packagée et testée vit dans
> `src/rl_from_scratch/dreamer/`. Ce notebook reconstruit les briques *from scratch* pour la pédagogie ;
> le package sert de version réutilisable et vérifiée.

## Références

- Hafner, D., Lillicrap, T., Fischer, I., Villegas, R., Ha, D., Lee, H. & Davidson, J. (2019). *Learning Latent Dynamics for Planning from Pixels*. ICML. [arXiv:1811.04551](https://arxiv.org/abs/1811.04551).  
  **PlaNet** : prédécesseur direct de Dreamer, RSSM latent et planning dans le modèle.

- Hafner, D., Lillicrap, T., Ba, J. & Norouzi, M. (2020). *Dream to Control: Learning Behaviors by Latent Imagination*. ICLR. [arXiv:1912.01603](https://arxiv.org/abs/1912.01603).  
  **DreamerV1**, la méthode implémentée ici : apprentissage acteur-critique dans l'imagination latente.

- Hafner, D., Lillicrap, T., Norouzi, M. & Ba, J. (2021). *Mastering Atari with Discrete World Models*. ICLR. [arXiv:2010.02193](https://arxiv.org/abs/2010.02193).  
  **DreamerV2** : latents discrets, KL balancing, performances fortes sur Atari.

- Hafner, D., Pasukonis, J., Ba, J. & Lillicrap, T. (2023). *Mastering Diverse Domains through World Models*. arXiv. [arXiv:2301.04104](https://arxiv.org/abs/2301.04104).  
  **DreamerV3** : robustesse multi-domaines, symlog, two-hot, free bits, hyperparamètres plus universels.

- Ha, D. & Schmidhuber, J. (2018). *World Models*. [arXiv:1803.10122](https://arxiv.org/abs/1803.10122).  
  Référence fondatrice moderne sur l'idée d'apprendre un modèle compact du monde pour contrôler.

- Gu, A., Goel, K. & Ré, C. (2021). *Efficiently Modeling Long Sequences with Structured State Spaces*. ICLR. [arXiv:2111.00396](https://arxiv.org/abs/2111.00396).  
  **S4**, utile pour la Partie I : les SSM comme architectures de séquence efficaces.

- Gu, A. et al. (2020). *HiPPO: Recurrent Memory with Optimal Polynomial Projections*. NeurIPS. [arXiv:2008.07669](https://arxiv.org/abs/2008.07669).  
  Explique pourquoi l'initialisation de la dynamique d'état influence la mémoire longue.

- Haarnoja, T. et al. (2018). *Soft Actor-Critic Algorithms and Applications*. [arXiv:1812.05905](https://arxiv.org/abs/1812.05905).  
  Contexte pour l'acteur gaussien écrasé, l'entropie et la comparaison avec les méthodes model-free.

**Dans cette série.**  
- [nb08](./08_sac_halfcheetah_walkthrough.ipynb) : acteur gaussien écrasé, critique et entropie SAC.  
- [nb11](./11_pets_halfcheetah_walkthrough.ipynb) : ensembles probabilistes et propagation d'incertitude.  
- [nb12](./12_mbpo_halfcheetah_walkthrough.ipynb) : rollouts courts dans un modèle appris pour entraîner SAC.